# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung engine với `crawl_domain_samples.ipynb`: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Nguồn mới: `links_corpus_vi_en_zh.parquet`, **1.220.598 URL**, cùng snapshot với domain samples.
ID gốc từ 583 đến 7.466.161 và không liên tục; số URL không phải ID cuối.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus_vi_en_zh.parquet`.
Run mới dùng nguồn 1.220.598 URL. Resume phải khớp nguồn; nâng cấp từ run nguồn cũ dùng `RECOVERY_RUN_PATH`.
`POSTPROCESS_ONLY` nhận diện hash nguồn cũ trong manifest để audit/export đúng snapshot của run đó.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
PUBMED_SELENIUM_FIRST = True  # PubMed chạy Selenium trước, bỏ Requests bài viết
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus_vi_en_zh.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
SOURCE_FILENAME = 'links_corpus_vi_en_zh.parquet'
DATASET_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LEGACY_SOURCE = {
    'filename': 'links_corpus.parquet',
    'revision': '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7',
    'sha256': 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971',
}

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
default_tag = f'ids_{ID_FROM}_{ID_TO}_synced_vi_en_zh'
if POSTPROCESS_ONLY:
    if RESUME_RUN_PATH:
        default_tag = Path(RESUME_RUN_PATH).name
    elif not (WORKDIR / 'runs' / default_tag).exists():
        default_tag = f'ids_{ID_FROM}_{ID_TO}_synced'
tag = RUN_TAG or default_tag + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    previous_manifest = json.loads((previous / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and previous_manifest.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Run resume dùng nguồn khác. Nguồn 1.220.598 URL cần run mới; dùng RECOVERY_RUN_PATH để nâng cấp.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and saved.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Nguồn của run cũ khác bản 1.220.598 URL. Chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
    if POSTPROCESS_ONLY and saved.get('source_sha256') == LEGACY_SOURCE['sha256']:
        SOURCE_FILENAME = LEGACY_SOURCE['filename']
        DATASET_REVISION = LEGACY_SOURCE['revision']
        SOURCE_SHA256 = LEGACY_SOURCE['sha256']
        print('Hậu xử lý run cũ: dùng đúng snapshot nguồn gốc trong manifest.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/{SOURCE_FILENAME}'
SOURCE_PATH = WORKDIR / 'data' / SOURCE_FILENAME
print(f'File nguồn: {SOURCE_FILENAME}; revision: {DATASET_REVISION}')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pO6TW{RP6@J&Rn8+<irn}T7j?qe58IJ8BM&iV9oS=ZRU`FKZ?nJzKNJ`pB0$SvuPX!vJKoJxH;siyI21Qe}Pu4>p6y$%Xzog%p8FINd$th~U(k^*!bMEJiR##Ww84O~X3MSK143#X$Yhi|uX~`nVhiO)nGAtNRBm8Go8uNV-$0A}%97=J-ijfc{8)S-c7Ak%eGgYPj>gwu>O!7>XjH_YJRUuXeDoa?H0XZyXmKMwc+|4TRQ#JhXW0}{VN4yxta=-anCNblM{vC)S<hiI{eN<$r;gIu^aBAw0@U;d~I7OB;-x}m%T+|<{REAk3B3|+pLoYrC)0O}=WIpCfWk)^^_q2ivAXol$Wo0E21CZvCOM-Y?D-8dXC@Ym}A^kYxQK4V@1QwLyxO7Ddt}-3=oT?mjf9ZIhr4(jK&co6rsA*yjbNr7onYt~J{WMbvkLAb0b>4pcuG3)}&oA;=mTp6|7;%@trwLkR`*^$g>MaA{kHm2#hoS&>eKvj+Xakf=2zNh=#vN9RZwW?13&eViUi;>2bH)yHl*nTq3dhsna=_B8tg-x}R5Euhsg`_iXQ=?LNb+(VkQ7lCRtX%{(U7GY&!Lc<K3S1yrbE}V@H(29h8D-6$V=vGP(f$%ACbGN`>?L+u=jaf3H^DgL{+LCobShCMlPeVgu_Y}TSF>nrVb1Ljv8uc!xbVHA<j<`(hG=G5O^(2pGQ%^EsE=OyIEDj%t}11q!Ll@iK@hv@Fr}#1$>0uW1Nc~BEJLe4|o-qy^ZztEAfl0QX!#^M4b1W_l1%J30s@z2saZAO2k>nOT5rQgw&D65e{C6qQKFt<w3ZBE-Wkki<lLzd5yE-Ek&?PPknqhFA|>00h|(Vb*K^G8guH$J_XvuxyapXkb)<9EHpsucqRh($E5&EkG<4uDDPRwz(j-7ZyBG<LbUwCb>7vWI)GRri-ecq$g)3@<%lg|p5-E)k9<lba96>T2}^^B)9$rW2xV<gXI`7XEgBt}p45TVuk{dv-HF<BWaRWUp3+E~_L)9CL74B}Rp-4|A)k<SYr!7eE9`m2agOw^W4~s##qgn0@dBnA#duCY8F9+m)}w&ZGK%rOeq-Cl<U;}Xw4|)fT4yjh4CH5$oYz2TL$yRP)^XJSVvEo}I?X=Z{D4j*QNrAqC;JiKve4SQ+>Q0ko9tyqzmRwKowiuh;`vpMq>G$NG$)ebl=giq8PE9Y)Vcf93>Nt7bgVA&TdRD;=2KgyD80EY1$Ed!#yE7*+jGgzNQ5%Q&sajf+6_Ff@A+jG<YWEVYrCSM)%ab5LKeT$cQoD^;ng@V5S&pa3SDOODQD7^J+wl3P!VOMJyXF=c;$wx0;496SWraqFwXW}=VhC7Y_y?dS3JHZ2rP}WS}7K(buLl`>Bd6N>*}h5Kd#`FDK!wWD0LiBV~1K)pv!4EAau9c5tp`yi(?j!M0k*+vncDJ4mr{`X{Fx|SgIrR^u!!s_E;0q2%X?a5n2zV5UQI{$3!o9@D#^K0~!=1FROyynXsODjxcftF&`4dUjM8eC=T^Ij+R2Vtd#n}KtJvKI@b!D>Fwhe4VuB~6F^VL%2s3Ev(>41wQ|?;vR)5;%W<hVsoc?$kt<p->KQ1aW;4SA86HZM&vZyl7(*eCa3XP?6wAm{Zw9kazpY!eBv&ZTwTFXbgV>)blv8AEfhB6SUI}rrT9+K_F^=;Bnr$_Uy0L_QRasExcK4m1Kf3pNcTsYMj;K#C%D_Qmjz%e{>#n2ojV-=&xYQ4JOn(xP%Mqc){jWEN3rQ!4^J3MJv1fg`@ADibhvtSF$y<oj4!=FCr9K36TvDUDJpWTVTss(D{0n}fX|W$Z{qg(D^RGj8@12L+>{ukN7-xqUUut(6UY>s%`rf<_L|ll);^XkrBnF|wUpFDTh1S*3iR2;FOJQk8%C3lHnW=7$sZvL&D~wrSl%lYvaGH6gzmO}0R@qPEiXLC(|2xMp7HD95%`j$tNoSP~p?k{(ZJ{+2oA7n!vEsSj<QV2`OXwMIvF6v1K{$ee9kM=^1^w(QT!Tb$F4UojG@_9byv-@0H}izdJ{gl1IJW_#TxLE|Sk`0~`^!?7w}OE}FsR~q+!E1S#@;yJaz;oEGSp48f!tf~Un7*&tnPGR9Bd^T&01pqG37(o@z>Tg)H2J@TPxwi>uJhlcoju7^#|JTx$(b|5(CGxyBBD_CTPNdn;@8iy-owXEO|JhQ7=GEHjcGdX$}=}FcMH{!E_<G3P-MTKJeW<8k}NOJ-&T<92dv8J@@w3&rbJl-{^aYkBY`q|G^c^Z9}L%$74He<JEQTSIbqn-0WBQm?oej&yLGQkcBER6S1>g?>E?kG!jIFwqmB^hdPC#>TBj(N}x!t9hQniOu0jm3N@L2%aO3rMcn<RjC2}^+oqiOhS>_7(x-A93;G}jiYNLLwsiXSqVhML%Kb=XxhY-C54^O(e304<R@TF%mbskuKE2b@?$-b==C-`HTwUqjH03S4_$M|Bv`~Z;uU$TWhR*&Wm)q<>UY>sf#VbCkZDfXw)lE&i(vi567GaXsr<$pj$3K)IvUD~mAJF1573KZoJkFF*r%91@jeY1-X+G|}^uZpSM-EPR^n=~g9qbf$?RL&a$Fvp}dvG=wNa!O|LBiERl?U*t17}gsH`_5&W9IOXR$a)cSMh>G#A}9*dD}s|-7@49BJ^iRqymUjuVzq+`aJZ-4j2^Gl-ler*d1uU3Uxiv!v}pG=&uv9W*DgwC}XVW3*!oO!1gAiLydqLg6cQ*JcU=*#}dedZZJ5s`z!#`Dy=i?o4U6}ZZdg<dRAnSStRC6R_8%IsYDVjfH|l)Pqy9}76G-hrn&<Or*lJNirLzY@G=7})#w>@y&!`Ywd+~Y$$m1Mc1bO!nJlQb9k%YVUABHbn<B(|2K1aeRhcC^)50)oNWtw}i*2nR8g=Pj0SicGhcIj)nRRX7nWsLguv)QoA!z%~x*0a>B>=C`&IZ;4?E=6*!X1h!Gp=6NMOBg)t_7k|p{NwLoJA;Q?{$v+*TF%lvh+5+$tJy72QFY}f~ejiw)i<VJGc0%MI1f$3!Td+GrQIE683fCn8diN=H~sYvQMv_eZb~GjC*ER##FNrsCO`0>$q7d^sq%YHvVJ%%e}dYMqLNBTT|;`WK7Mz&|=bNyJH$G3**@YWUi`utXbh`cr3iFF58z88WzbATdSgd7Xe)-x(B;{n^DR#7Lz50`AR9LnX9+vNA%8uVu<vmpyU!DLn3Ie9csr+F|`Nl0k&+Lt(Vkp&=z>^-694F=CY8-L0TpI*yv5=1xwIqOQ^{cs7c4P<D9jfp)ud2-!sbmzc{Ma|58UK1(9)_otKb^=yF3}0qySSo5Pp`e)UkU?(QU_#C0VNPonItY7%&=O|*N5f(#`|<yEn>W}eL?3C9+4n*L7)7~{4!Zso^xj|lO)mE1<G4`sTwc{4xWCM$GDLJmg&^KYF+;4S0gWFJu@)Yb+eX0eRejW7(ib6mLKTw5EPK(%D(Gjb0uNTfQ6BxY4U-ojmNv`s&{;E5-t=;Cs<O44F$W1txRZx4B{ML2^!PmV^=UAMp_Dz@;wd!%^2ouPCM;_OI&+v4f?XoR+Q+Z^d;`$%U#6|xhs$kf)2U)_4;Y{zt>^ST*r>^>UF%jf^D+I<+}jFlH(N(>Q~=fA6(q_VRH_?>+B;pOx1DwYn-uaUg?E0(STN$2LK&+cBHe_n#j#WUbzVWB&=^78yU>blX`U3`_YeIkxX>ab8l=cCK#UmI{^OrQPmhl{Vqu=|hS^ZNAx<SbDP@C7d}pZ_Hx^lvyMp{2<e65ErDXX%Iy<;639XD#1#`m_7a#^on@0=IZ{+c3GP_U(Oj4M471H$sEe9qSakJGM~lHv2F3pWpnEodDaNN+f3QKt`|#P@#dc8HnEHyF1v1Fum*OIpjVLIGFo)1jQ@6q`h=lU7ml2&B#QJ9jhbs&7URAu1`Ez>J_k2!Z2!g=UTE!F8sAfCVjUNlh7b+&ATncU2Pp~W(*(1{6vyjrt_<pdD?3@GySwGU7h5HrjrI$$_q2)wN(-5^-OVXh99D|GzBG*PCz%<n-ka1?4MW|H`H?D_GGn-<@?%ukKbYkqs#L@=d~fQ`)v29XKN>|X7<cqZHJrFWwi7=tV5#N^tA%--8b%Zv5K8;bcp+XtM4IT{|;2RJuc=dvfAf4)A_=r3wC0iZ11`@vJYvmge8a8fl8yBHTd+ANV3;zh<$`5<~$(c=NnHRzSUCFGCtEiXa8)Te4U+iJJqPWiDK_;VWQf$K|yjg$-$%yuppbZ4RiP6S%@5S@onC=MSAh&m>tur6o`LIS+0<RQ%;KnG9nP3;gW$Er!3KFtUwI07+YN$Ywg$zfB6=_(fW+YZqL6NTl1wrku0k45~}+Le(AP1NBqNS3*VWO^OSPy7wnY1OMKerJ|*B&*6ntg{<(!Gee2`(m^c8y3A(v6fSl+i9-mCP#?y`&>{{1nX8!SY6?f~tY(j320z+qy59eCl5A_j0(Q15fZ<~EM6Rq`o`uY2?$!dp9`bS=WPKi7n@oF1jIOT8Cc6KM&Y>KX7Y_!&&&J}mwyZ7Kp@c90tcOF0SlW1A4NTTJ<3jQSJ1!e^O7ajD+H|R`*AjQTf2y{ao1oVd`L14#uRBG&6B8cZx>PQE22qNy1ET#V>W#zxxsz);',
 'corpus_runner_snapshot.py': 'c-rk7U31(>@qK^AoOzHox0anta)5&#OcL&{$i=~N?uuHrgjSl}9jujf8p)1v{2$<f;+3l_iYh1yipS)Ehfw?v{}a0BLz>aYuAKvl2Xa;C-TCOA?&+`T?%mnhIj&Mtt|;NFu%I!CvvtT)k{8*W5}GbqN;wJBnB>(wVSGhCd%T+$l+)sjrArcJMP6|dt!Q+bXDltbx3jY|V(UCBN)i^!JS;dJEsAUnYm<aVCCgGy)PhG@m6o)akeDvQDk)<Ym3ni;&-6$6Ct-81Dp^vGX8O}Fc$VtV>#$r&d^voAJsQ_@oKT<n1s{#Rcyj#Wv!iFkCypdzx2V#T7Q1KnozdebKRSH*^yCH1UXBR-b1I-Y2$wW1ode>0p8b*~Nx1jG+b8aqERC~sPM)2RyZhe$LjnUI-+xG6eSCjR4)Z*rU()%HS-JP%-gmuw9~1Y-pPqdFbV3q#O35d{x@=4yt>Dab@9sSSfd7#f;UX-U+Tu*)A!3?@7l8k+Hz7_~mUNw$9KPR!uO(a4tisbxKiY@+`wzSs49}~0NpbY<_h3|lfeXyxDgwCWOzi(rm~g5m#bA)-;;LZsQV+Eds?jI|t^u$H|J?@)QO`fUtDs?QY`KRgZsyA3BB&4keRFORvZN|#P?m|@cBPIzFW?H`iq=_7@rH0Aje{&rE}Tg-ZC&HDETu3P3El^DmWIWJ9+pr+?AHK=(}bq1S_cbOaNK!PRJ4US$t;S1{RZ0ekd*)hcolQSp=Q$H>qnE()o3&VeISBEJY0kLa5pN#^8_^ofR7IZ;&qs^1?44(sG9Y}5l|akSVs1Uqt|#i=#0fawy4)HfJMP7<SV^iq7$nVXG}iuYb;YW7l5Jo=)uR%?4Z8kB4nJBFT$jvPl_TdT<3*YN)R&GhA>^y*i+cgS-B$p=)5easY}x+i@~<|PE{^;zXzuXIpHO6W8I)sYyr}zxLFX+-2|-Fm@Kk_;3oohOzuOHam0hKEO*B<%?G$t_p+EeEOusMUm-S0;8%ivT%rcXn@E0K<q3<zlB(@sD7mH?<hd$pxaRd1AQ44u-R=YqlEnB$u2)#mngO|zadHKI*twLeBn?-Bl{=ceG%LkEkG^8O<gQ78JQ7N}IwBsVSCkM!Ejg}S=z(Ush;lq3=15h*Jqo_L=D9R2Q?X#SiIEHEQSD5ChkGxNpOQ7>VAi7*m?AvPmCaU(UvX8rsSaa>LsR^<_*&4fDyc-n7viWzTupPQ2BP_*!7wg(I?QOL^PBN$!>|pVWI8Xyvf`j9KDaV|%JRHxv4juTxHlb=knfOVI0bt5Fu?&>je-<{1IIIz5G9D?8eZQMkisaUoKIBPFoF4u(-bHb%=hGU0I&fNbrlH{eX6l7hZqXD5`sD7Re20b<|<&4bl=A8%WaZ5a>&th<37h!U#|k!%iT${Y*m(d?^<KR*^G&zum`6QIM@)HaSNju?pe77i{x2=O5-nf+ETpNXFF;UQ`ce&iFpB9L=_v<*EY-aDwAW@(ybgbbVhN(+ON4lY?aX@w-l+D;vuY!28Bs752I6zSzv?j3O#99Bu`SK0s*I58EqvTiirmQQHaWv_Gs{}!)P@ckL!z*f^Ok=hHg<^i9=tNMHqo<I~}?#>r&7Z91kuNM^g)@deLl%mk{@q5OM|OMNWOE1ay0=a-BTH=l~uFr7wN`&~`1kwg%`?Q10mrwM9gTtu->j9TY7%cmZTX2-eYhH9ou-V)2{{Q!BzcA8tO4U0X@*C2-RHG5OGDaNn8<xu-5v`{VP1mDGyuv~w5?1GH-%_LdT;vZV#Zx6vk-sUx5~%c~@mFS!lDC(o)Pq5;6cI8%k7QwNN536BPN`GLTUq^6Zsz6nxoY;d780}7moNw%)410X53msgS_upImXt`m{u%zB4HGzn@Y7Gh+mqPiwLY^&6^2%^uE8*!Rka4>3X4NEK9V9L(%L3&Cr4$QC%zRR!|KHv#m3kr$^oB$&)_#+?q_{nmiI8eJvrx`Qtv#kftS5=s>^1=Y1M;1xAln{k=)*qx>Gs7EUD$V$+A5)*>wYJ&d?cp{s+i{u4X+gKVjx)aM+*o!#Z9|)hI7`gf?44mu?0j)}6g+-;^hkzdLQd@)hn)iswokg9Ny{~gxv66NY;4NS)B&g=EC@glr24_3d{dW&ENMHU<Yb1{$b`ea5aLU;bJ@8i#M_CsNtJpxZR4AD;0vT2O%sT>pM>jq9FpRIxVuF!QDFAW<E!VLGN*aLGBA2fz8x?1x&xK?LIvqSJm@e<FW&F>4LbJ#Ab7yi0P;!z1YeR%5czyylVk`U1+9-tjk!fPE+qI+ZG3L?aFx&vNp1e&8hvW94A)DH9{7MY*RSG+LGR6T#cE!y*N{&^gCznH5ndcv4{FT4=tqT`!rR7ilmU2yv^J1kV^Nz-C4Fq3U;+rxyG2(beM=2%)(t>3<Hs`*_|$-vwVc_3mel_(Xm1<`3Sj-=<p<sH^2XG_9j|4k>%T_J21@+*phPN|BA`($v9B%OXP8_7`9`TyiYX<PE8%KHUa8J-%%xc&MO#J}6D{f60=BAx^dL=v##ibD8h~*E8E`O~#KJ-iIq$9xetVC83!h&5+S<c!_vu4^ynTindiD4A<g+@aOT#q;xX4Nrmud5&2+nw&K=Ggu;Y<buQU7zD(^Zx(gR?LT;wp>OrfFeIvhb5zyyR?bq*hh_(uZP)_X3ia>GHE9H>Ky8&Ga3oalk^_FslTJy~p4{z7zwl1Tc{jh4gY?3Riopl-CtRu&}Ft5-D(zTg|i<_OQ&>EDGiqC1{!ycgb9d9~EDyp$nln=69(pv%9;TEgj`A5>abHng0%X^zQ43oYOf6v?O}>x0EE;Z-2{3SyUG!x_<NL9PhO)JI+7<QjwCq`=cmnWu&}*`x{7OVG1$2e*1@z;L`)o1#sZDh*Bc(i0t~!H)|4KzxhMz^+8_j{Nq>ez5%G$Z~sg#YZr92cWHX3D@65>oW1*9M*i{ZfByCQ?Kcs*RMuf?x~JLILq$<Dt!x^+-r6Y%=cFdPfeTpR^?d#2_a#Zh5GL#ZLyif$1SVG@Bd6DI{$7%=Dj<&($e1NmrRbXc<mKVh&rW_C96xz}bbR8i<6f$r*W|IR2fijJ4KEejU=Y9Qi$R7$paY{%#mKSJCGxrl?C$O^5&u81{=nKsJ25OFq86!*W1|;BR&=De*m+GZadi`dT#;s|8?6|#J&~H67G;IFTIbxAD^++l!%QN_3RK7=#(YsZgPVW7V@mOdD8|UrM=RhB6Swf`$;tDHV%Pfm&DX#$yu%mD&S1I}CH7B}%LnA=U6b;2c}=s?7Tt5ceDcHK$?@^g@e8<H-1D>4@qPiRT2Uk6EiKd<sOuC`B6XdjO8avH@H$<v<%Haskn^lKr3LrzPoQ4(DgXoJ%hH;Po4}!nT!0!)vSmPV*uS^GuW|}_a9<Q@kp|^T1qKWt@7&P^S-^Kkj$hT)Eb(<}R{&??jfRh5CtEC>Nv9wvB8`BC(Mmbv0AdtfUKDNvGHuSrn$OGhG|kRaVj}jMDjO!dTz$A!$#pi?dG&dgC7Y1g8_BwYDg)H!aoZ5C{-Wj0QTFcd#cU=W`U{~$HBf`G>OGB>dynx2OF`i@EdYEb9z4c(@&ljjx8BX&G7@G<uIfWesIrqG1Fyo)25+iUm2DV@HL2z-JuAr%$lXnxZSSbh7uLr@By<USt&Z1}I)@9wl8zWMtPrY+2N!4;hvzCpxT?(oV?bJBSqH;jrMshUbzZbU=pyeyPo+2r#OI(V7;@C=P22UzCNRRvYyu)v;x&k?rn0Qp0v#ZS(W%?i%NvUlSjAM;{SnAH8Iqf_Y3%7Kq_!&RP3qFvCSGg$dJa1DcE#k8YO4WzvZsuoDx*In(7*uF0w$ImzkC*)9DX8%`uNGCBe-es=rH*F@Z?j^2o7mtWJ$$IEn<!Is3@KD(rxFM$KEo@=B{%`?HF%SKj$f3hS3E&kAs_@n3Nf*@?{Y=Ms(mkWw3Tq^R>I<<srtyN>`1(k6CBs@zD>DPJ+Wvo<Ijr{}zu!c^@9HdtlJ+j;H%GEA?f2Ih!|9Ryjq3T8Rj7Q6+(XX|(dhO1nFbQ>XHSm=%7v96fN=HIP;t$bc9_;g^E%$IGho9U`Pd%|eifm*FD>Vlz&l2Z6}b;ms9P1d3~swu(KQ1tI4n6H6X4u%cm_rO+~wAn7@P4;T1^2~%MKi=q+@NMIbxb4q2CnksEIh3BR<EfVRu2Je=fy*Sp*dj;x!c8`I$2GtTVzCCbVo~buN+f&!F^hY=NX2igzYfkKXAvEhP61H5GElF#H+iznq<$iMGT$R;CRl~Gft8M|#3?e!wv?2^6&#e-e7`+BG4dJB~8~QvD76}>J!sR{;lNuG8Hr?d_{Dz)KsCwDR+z#Obim#6Cx5#J$YxTFwNjp&VhXPMsC)Evp%|g5HZ`Oz5Mm9(ZuA-n%KtXd|<AHVA<o`6;9CSRcnjBxR++ax7F8;NuqIPdX<LHEWP_IjoyX%+Craqvg$4jF*^U`GPrEKLbvoq%k@`{V3;wv9}0>=aDD7~&aWIr{wK-5C^trYcUN_iV)>&Ih;_8kCL+z;4v@aqr9(fwFN_iMdN_QC?ZELFE|6e@0Heg`hSd7jpvy4MN<9;O%Wzf%McdQT%`D`?ZkD^StZ^J@M%n6^4~fDRTQ=~~J|_$-v343)%XK3;fxucf(4u>+&Uk~e47qq@7$wo==yJHDM%$c3pmm+OW=VQ7amY}R|egsmB-40JP&vWvR8O;-B#;I3(_+lI=9_(A7!+1BDZcDsQZx47A)O4Q%UkWhD&WSesI*a^!%khXp|Dotdob)%26^LhZ0c0?B2CjSNZv(*Kerl1w1p^d|)sVbEjyUBJfS5Q#_ZrK`Y=yvP=XM-4wLV+4Z$<&DllxTZzt=YAj>W2naT7%ZK+mS<sWik-?jEBO|_I-9$1=42Oxd^<vt<?r?qJ_e0F~ij6*b|*s9G@Jzdn<6pqnni$>t3}w_s3GGJ1~*p6?L?fhYLvE#xc;rot{#8m#?PzYtI55%D&^QqI}loU29iQ20-1;{32i`<Dko@!@KU0RjE8^T8lRv`6dClx}Ep)YqJ}~d;PIix8c%^0!5|PrgSK1ldX}RR*D-nQp!7+D(YtA+w^P#%Plnu9O(7y*RR`vZ32ZB`91Vs^_D08BN1v#u>K#Z{*Ohfkn#o*Z@s^6kQRC0)QFi-Q@y162Y}u)tUTvoMA4@Bn3J{lHN8T&Zb=;tni&Q378=o8025LLS_^i_l|^ll)0Ip^XNcu_>lcb1jt2N6M46$6&_iUW%<b|gTmou$zZ{nk5SEMoQ{{jQuBQ1E!C?=d{9d0&B%$FF=qosdI=K8}mHE^zNhH2<-TJ3Sg>)nJ^&J*Vcqydib1~z_lt=JGDkRAdpn4WT4&K=5fqM-?wZ+8_FUUV)6%s0(4DQvR$|SUoGJ1FI?#k+s%sE0rq`9eq^-|kzwM7NtutzkegZ$Jj!2bP>J0S?(g&^42xp6tD=dMuCUQ}2G-LzZhad&09RfR*aLWF&CkNAVTxU1VHv$~>MS4wah05$8!u*Q_|rW#UE(e{!xRRc8kDKx9EveL^h9NkfgKNao=BIXPzs+4~bs0vR9SYNRjY4T*nxl?2j)QJW1_OZ!Jzp%uxerAo@uNN_^b7`GAwV3VX;2U!*<_()W8bQny1gMZfAUfKDK-822m7qnn<z)<H2(WloFiG7XJE~uHqB4&D1#$kiJp',
 'crawler_snapshot.py': 'c-qZ<ZIdHMnc(;S73FsLN*b%x>Yf>&-5wc_#{*2*9(%`Q?lyWvQ7x5P<(5=(RjFU-=?^TyE^zn)4wits-B=rr*bP`PyUTHlhdVxu!~bI<7~6chzi`i6W@T1oN$SDetpj>WRhdt|J^8%nlk4m2Ux;6blUHus^7?6bG~P(2;}dTdkCVLZ-Auf<=23d$jl;cYoVC13GN6C+G>oV5bl*!-FP;vfV|Z?_udlBS(`4cW!Em0>(<lhMcrr`U+zY4EBoFgAnPzKi>Rq}&3)3v(I{jokj`~zWRl1hUr+JjNyg@V!=i_`3_j6Tv+MlOsG|k)GlCE=mlt$s;wImr|KaTqIJb~(A7W9+JY#im$K-bTs<9r<Nse1Be5>CVYDCJUvFpu(h5~(ukHw+*CemR*+*P}@okK6M+9_wz<z_5IfCxNP`2EPC0c&465VK#zpsGs}~yl79NJcNp&{xS;(0~lzgUc8wlQ}sLv^AY!D22aMYUc(RaoQBguGSR<ucd`+7UH?k@2T`tn;i6@E+}F)!C;Bm`xrUY3FN{IXr*R*Ug@!;qo2O$O7tM_79=v-qiKi`m$Y$d>w~D68UXoM!R8_czuRa@(qu1!Q#--Uwn5M}QEH%tP>17)if1XG5v0wfO@Bzhh`uJ8cIq)vai;<?`Fdm11X)>66JWyTQkMdwR9;~f>;rgxDZ@hZb>w5m-)Bf5USMf=0u3J$Sg=v3u4fj}-Zg$#R{@SfqUwHL)aP@Q7Z{A)|)?Zs&gB{?_^M0ebv*y8nX#`!FYJ_Z0lcR>hNkGJYvmIy2Fi8O~8cnG_i1%Sg4O|c#ilAUS8-?4KpW*`TQFJU7RDGC$D8RA}#4!PJ6z3yCGJe@kX3@0ar+a=ACv?b#9BmOWX<*p$d~Vx!wHxNiB<=@$CwY`nD=p6$1HOQ{E4rZH;>sd{m>dk^w81~J?(K9A`xod{mIuiJ{p2cnG?@V)s)?ftrr{(akZpYOp7e%(`&6TR8<#yE0L(`7tesD0egUR^l*X|0xyRJk;!_&U#$i8_zVZmUIPPZ|U$Xb649_umEY}{)C$p>p<zOt+3`uR6_2anv*>Ie}_>d$|^X_)D9Zhi;HvD-$+<4k=swT%tI4D}Q@qrq^0$IaYSsZtOB(gyuVG6ospjdG%y@aim)sMmV+qmx=##mC<NSZM1{++4s0WN#UX&7$Zs749k4G<4|JPk!$=u;aD(TV~{A24p597U<dwkd9!as1_|fnyaw_z1XRkhFGD&cFHDYkrH%2uQWa{+$#$-=ZN{Z`C^!dK$H}`CcRSckkrhoq4Adc6v`@K~JDJ%|YqXcc!NoTW1VVH^#Gj+|Keeo;8{@eaAR`2&c`HK^<r|RT&xo40;J$r2))YtzR<F)eghAZTQVyiHBYTKW=(H@XdG#yS9xh3S*@WmML;Y{SITG`(Fsh^XNJd-n8MrI?sVDd*K*S>jX)I7kUcNvoOLrG76g1{wPV{N9awXBX45^8g0wuWyw74M^^v%l@Nj<?&-JMuQ8QJ&e1Tx!w_b}cGe#O4-c3jAZ}#dV_k38j|VuPfHi&(djTR0(1&(@GK(@Gut5wW0|*O^GK66qjRy^^-K4?`XW{zs48)!Y*a<`P;sHoY0}q!4<SS?#MWgv-nyDejgNz5oTQc0!V`{0QUFv(U$^WR4LdEuUJ|2TK1cDWgXx4AjGmv6n^3#6afHKWC2u!mRKpU8ZG0<~-w`d5j#91b?;9w58)d#ku7Usr_y5H0ic3tGU2EQK0DbV7YkB@tA+<Msy=lLiBB6;G)nU~Fn!?+&<eIvA<r3o+-X#QF65H=*TL~jU#Jn{A-_zpj!ygx#U&8<O|y%65uZh(aY%9{rC0*F&Fd$i`pps^Y*1yrKKZ7$mAzzQcYS>qTvR#9GN95E(_xdAb13OgSlv~XdY3Y%aTPY@jJ<1dc#e3pG?XLED+i<`9Z3qof8SDF5ye`87`S5po%w&R^b?Z?tHnR@jf$bTTWBPmPAiVSQ*Qx}`N+dWMQ%^JKeyIis7J=yiP%7e`2lZLs7?L+JwZ2qQq#oOw1I@Lj##jCIZCU0aQuikS6zXBO1uqXq~EW%GcKOrgs9#&B}`M7uet$8@!K&AkcXq@y(NnyDrjc{9}@!p)?Ar4@<jK2<n6d^Zayn1U6`VCAh>L_g#5f7r147OzoTU&bHY<V4S^=K4hKVV!`&6+ShDjWG^<=}r*rGjbnR!`-X^12a4S+za&pAAzlPYPSTE80Z+7X2fHH;M*|Nr7;Fn7}IIloZ-q1NVs#?kJj8X~Cl1*(}dU9|W+L+LoYGH5P2mmc4JctdFH_cK%(0gQ?#mGtfqf!K+D8MXgffcoHL{wPwA%%t($A9IhNysID4*Zih1v#0QO2r2jj5s)(%ZctnPLZh^ky!<juI(DEG^L8?9u1;{BP0Q`R+n2#~~X0^^xs@3*b^?H}e^#J~P`pwwHTf6JYj%%dM9B}4QYSdG&ala5LX+B#XY}oENoa_xkFWvD{;h+pX57QhecY-PcNd94-r-bv+pLh_^cR-Mvpnd@)SAk@vzH8BmY9U9X*T`%e)Pj1mh@`y^LPI<R9+v<m+SrH(8w?Hix%>o3jp(>Po@eo4)YQVm(9`WA2f~($hO%hP!bqKb0jw6bByh-v2uII#{T4%nfvcBSe4eak&)Q~1Gv(Dfy>KBlmzD(iRI%Z+Vy#pM7q6GGS-2^h-5N^gVyG{YG?oIckXU+ZKsqC$k1C|dZ5tcJgHAR0v$nx?6<;{zvF&)B5;<o=eP&l^&u1veG!%7&(aGHOjJe<rHI<OQs;0|onh5R;gv9nSt{TH+E>)<dsxV<ez#<n{f;y+V?><fqEFVyAWxfJ|a}w~3;YOIy@`Bh250<PcpTlGM93I=Bi#D`~38VoV>p)dv%kYfrq?s+f6(Iw@@q+<~g%A`_0=uLsowU>xmCDdEF|f0`>9Um~YohwFw*Yy9Fh{zEh8;wIn6E6tH~!TdpPQ%i+hAio-c)8G+*x3z9LD=tqMxL*d4}(ZL*Xkj%KF9#dA;BhNHMa7VN++x_%H&d*U}3r)|VZ_#v@9S2@IzJ<Y;UOmP&09-$_!4e|Ua`!WGY)Cc$P073~CQLa3;-L~Rw_3^Wm=Z^;b%4Uz`@(ozS#V#**`Ac$P<zjLgvtzgW~BBVAE0Op}Y`ALNh()m=;{(|y=o_!c6^UTs*b@`@YH-lG5k9T{P^n=DuO#^I#mW+vhs|Gf)AAl{eS>SoNehO8YSf0^X*Ih~rz;NjBg;FpDZE+sibd(x_mJ8bwKxvktb207!Qf1v66zSDRR=L$z`n7^4<liEb!HpL;QE8mS8R=GSzorjp!UJ1ncod*f3N4nvJhwX??@HGyfr=TTX-?a%gQC|(mr?_e?0HwC(M|L!b@mdle00ep&>lo+s74ci18Ok$WusZsDOHJ`6hRPwGn*4E@41Dox|zcf;Et%BJC=_7&||P8rD302Z|SDVivs2Zw0iix;{7n50v%v(UY$3dYhif+epwMDDqswtqSdm)akw9#>IRiSh@hI)Qb1#1abY05`Bzvfu5nJQvZW-Dp9M6K*$L7fT;!HOT{dE-0s-okU8Lm-$t-FPmb*0R1NsnO=@M6|0<N+tgF4{0NBfsrIaj~1CrZCCH*-<e)dUi@3P02oc0Ls*<s6tV!)JcaKZNGaz(|&xX5`a4Mo=dtKo6yDv}>Nht08ckk*wj`T7Mj7nMbY|-YeIS`_T*y0zw$kY8n6oi}N4=RcJgUc^HTV`Ym(t40gdJY>C)lE~Ek|0MkS9aXv%cvaR7U1SE}Ivgpxg(X17vwS9omU<bzGNq4IaG7|75@IL^qUTk?6&^`~ULl)?_$evnNVj;y~Ur%9D1N1Qg;O^FC1Sua`O^`1aJMg6z4j8Y~lVEQ?0G#hW(=mVyVVG$A&+{?CD9;!Thn{Fm@~pbAxF{~6jJg=H4X8A5sHq>3GnBrHF2>?a>7%J?Ai-<6;5U_26tl(E*$2yh<tEVODw8=FtoG0(K1;GV2kA%|kWHFxw#K3(Dc2=ni2sgmwVb@8_|JDE2X0a4hgIR3_*4Kh->wd+KsOUo@pe^626`5NjF)GYa0aSr%OgXB=&YqhK)co?bJ&XwTQOA!s}&i0#9Unc+<-+#ij{<UQy0T%0Ps%?kse4%`o=v1>87)iS@{%(ai$TCp^=QFFs=^MWR(Eb8a0^UT2wbIYq6{^#_Xd-%2m8t(r#ma9+Jdbcy(Wf=!ZtBCtwsl$fWHl$UU$@?4Fnf_!aK5vzGCH=h(o65T_?pw$fraw)#bH&`GDLJ+>JDt_ihIqkS}0pO9I^z$>y;7h^3^8Rbgkx}ZtrVykTs2A8MBXg&im16UxnZZs#_1IUo11{H%L1&ylsC9E&Xw50@@N;&?b{kpW%Tv7+(2EqXtrj)>R>c0W%=El{15YJ%4zHt>gaJ7$&n-pbO#m!MZ8Mi==A!7o7*qjZ9&%V{|w4Z5R*t|ebm;JM5If9|uHqXlQ6e`mP6>*dd^gJn-RKs@WaCi_~m<!jmIgMO`{fdz=5RY{c)cMI-<TneiB$YN@+wuxzrL^>F2I9j}Qb7hH0EV37$LuyCvE9`-ihfZ&Bl*F58W{SeOBb8x0S%*p%1G7*xEm3w(cI(SRal}EFHe+v3n@@^h(T#e$cH#Je8HYRkJe@r1j9pg1%i}117CP9ZW=|QkpL5Irf4VB^Dw$9o(_{nX)asVaG`1g23TF8ZdWY|%L-3-nl<h~701<%XMsF6usge#db^z-M(zar<76)!*AZ!RBInaA97YO`-zsB*v1MgX1qrebrW#~WC!@R)6?qviaD*GmbNpfey9;bTQMY8ppv_?6c<~9jMOQ^fvFj5Fagc%TGtI{*EMa@%RI@81bwlY`kzY;eG%*I~7EY=jDv8aWJgn6Y`)|zwUkq9jjb%}+?`r61gX|Pz_cmx!+LHuoBoGP$1aT>%!>Y^xxgpj`pdh;6;+622O@Z1t3W3Q2Jva)1uYz~Yun=R<jO_qq+HMl7z)VSBO&&$Y3!EpiSvWdh#!hnzwH*(48@2IVchO9&f~)77i@g&y1$_{K2FcO1WQ(9KWaB8BHL!!FkL7vNZX9H*rOrE)T;3{$Afuje|3BRVE0NnG<Cwu33k=4LQUI+rt<*zj1+rSHI&Kl}Q<`~)$){#czz`ob(2y^}Xwkt<Z<`%#S_m#xci1Jgq@rYtCSfoU4z(pqP$-xp3MXQJLI3@cHvMEOgbW-wY69Go)*sEMNK|q#>5s|O3R}ima!9dAP+FXd1eCqhV)|-4b@}qer<(Pv!;L)2ku?OXSz8trODg`{BG+mCx+lOD(IbUUxuO3GM#JQza0>n`gm&f?7Dw7}U~2Ricc!#)6vjgTE?uU*QA<NARSkO5Jny)w_*~y@UR-6cS}@^@DBz;r|H(F@C(^e@w3<;WYm<n;EoD^J_{jwVj9qLMqBU%td7onE+t<MDbOTCGS+9FBCc~-J7O)sWp~5zKPTjpx)LNL5X|kbz5KTI@D<Fn3lqfH4KhyFqcP=i#4yG=~NR<{qcKchX3vOH;!tTUr?kqqOC%afMD)m!TZ_%;q;YVaMG}P<Gtu5`&!Y(<_9!MA4_zaUgnKiUbK%W<M-WWim7(vUs4K^=%d=&@{%)}A>Y%cFRsvvG$HUyYm6kd7`ehopvYP2=1iZwOYzF4!v(7-J*c1a4ZZfJCN30N&xFBfcpV!EB3Ab~UmOE&CGTkQvf37Zd;oi~m5<Ea@ef?rXyLgxm3#31TTuGG$tbAQbeh)55u6^S%S?CR1_)*5JvF`*^~_Ie74;~;<klsj{_bE>f}n1T`P%va&u&F1dT)>DP&1zZ6T#!k__r#{yP?VTTVP>>!jyrE{xUUk(Mbot7o+v?{l(g-C{1n|WW%1(cYG@#*PNDTZY`Jyo0%n~pqz$!<$382rcIw=bxP|PU`v8FwNAwf}sWGSf9Hpv+jWVO|D+m4^~wyKKe`S_KZXJtI00d)OaY|5g3R;5H4XK(;PTV3D8f?$y-X^tADwyH-IxZc5<t9!ep1?d&G`?b0z?1{AoJ4<E_zZioRo@H1cWc-Bu5TK5yo+80P5X{8f2ma<3dVVp)0Xep9p8y#og6(X_)$;K*Ww}HIUar7!1nR6niaJ3P+9%kjvPq=10;e%}f}aSdxt|cIF{9wC85LA}=&Ys*wJLhs^puoE&AS=OUbIi4l}mN7rF^hX?BQKv8awppw>sOGybJVCYblRf`kzut9Hwu>VxooI+6H9+rrC3eY^FrAdU*pEKy7Cuexa)-&2-dvEve}mf7w9pj$)JFTyd(jY%Acprw2RUA^StX8-hry8WQ&hQmO+LPYd54bx9g3?qDOAPOCU>Us3FhP>F%KMd7J=w&Ji5t0bJFYerl&7l{2w${#7je-mDTv3652Lna>u+rzjV|5S0OvmiW`&3P82r?4H*+WTMjYumHVS1D&H#x!7j0Y#SRxDt-mm>`yRJIy8dV@5*|`%ft@1tapn0d!m^GK#3E-Q=i$!@Hf;kd=84wgq)S9=P1;^jZQFfCc3I?vlF|3xKB_&748dT&#t;wfQ>nuQUxHwaqa8AOm%I3NZ*hqS|vHbPk1>2pq<WFQl4kxTS5W<7B#;W?4H`I#dd?SvxNt3mD@S1?$LacPp#Ein?~O0}{%`&NlwJh=2Y){&||Csw@igvf7*r6orC4$`Rp$hzqRU0#0v%_V0>Rs~0uuBj0QEd0VzaWnKwQ!umO?AMrODI0Yh@MTt3+$*hSkGfoQPx>2-PaRid+933z8YZuGEf-Wy#9Rmbe8$w_8c?LSWc;zvOsUobZDYa;?D0#q;sc@%gXAiZ09%D&mazB6jm6ttqfddM7iHg=Bou*lHssKfD=tSw<ctZ<ai|X-N^+s`K68U&mOZW5yU<DdFJ~;5JWasEy0;kzlr@8Z|8^W$*i&Xmo8~9SKy`^@2ha4bz%An5t^KBh~8H;^VWa-b}zWth~p*7F5KMld(8W>pY!J0%Uez~;Qp6Yb|Uk<lG=GDi`CKhvWtXWodn;QMbsoI=d7phUtR^|3eD~69M{LKz{fe$TX%9;{FUZ+00qh<(5L(9=(TGhP+OV#TPy}2BP8l{jMf~`SokpzMiUVoW^E@#BZ3>&pD%iV3RC=ny+Hlvi4SKg(g?QU%KmQt`1&R4}PH#J$bn&h@G)*k)kRcm++>8b93>V9$>6BaY38l%P95(ruvE+kZ!qrd2`Vb1AxomQNk6?R(5p|0hjMNV}oWH0w#*gr^ycrbq3yKup4sJ?B0B?-()a<CJU9bf_x^eVTvFE{I)&I0U@bUi<khuOq#Tu88w!umwAxYxu&ys3uEGFWDKDbuQf%RiSr%HCC~A2=9?u7IyAdO@udyR0Z^Z0hPKP>XEEq+6?$i0Kxauh$m(A=s&cQJjn8v>f@Q?VHjyu>cwoCM1z?u_D664121LcoIW3ve(m@#MGR@S$Oy{jdD=u4MwY?Bo-Kv3TLt%Jz@v!RZQo~@*W6+J7!WEU#m>qpfI=FF`sS+;uS|EAZ;6E-sT|A>9FadE!<*_--2Q8GN4`&MpJkWWZ-LVnX$6MUe^WK#fiB%M`&4_lPu;nr%_6TO-(=zU7P9HCbx=2(V*J#QjGxpV<=U95T`NQ4nL3?c<QDks)M`-p}fo3fStd19)Z|_VkqgU8FtHf#_0{im?R7r4~aDDtXSY?lKd|P3N_b09RxFcL6k_KfKZSv-W9E43zrC^evi{w8&S$s%xF(Nu9XpeN${75Et-;(@k0sZ5lf-v$XU_a`r$Mnb6~?r5$4JWQm<FV8HK};Ze@{Lc0zV_Q&=~kmu6$M*E+@tblRg*M_JeLtyRe!4pl)?fDfulNzze$!Tl;eQYtO$Mn+7da)l*H9lb)D74hU5QI2?8%?vcL3{1t_5hs2q9kvV#kIU1g1#r43)KF9`c!AuCJ30k3(WQ#jte1I=Gz3<VwT--Z#_G&L<}1<1&h98wxA0fouuYU?kyo2Ru*|UejpEuhhAdo-T}4xRu1!{NzO7c~$F;Jntk#FRz<GLp@7W?pnd<7Q9<o}8*CRQ*#MPxdUW}H_wFQ=LQ;F<VkEfNRK~~VS7h3<7echo|T`X%=21b^Zsd_D{o=gIMLJa&tpy;u!&q44aCH6e-uhn_}L~17$N;*&%5LIJ>E}P%jN`)~`9sILR(NFm25^K+D*jH8=_2b^l3CPIo?b6{gAdEr`pi-1Ds6K?$uHYG?^PcRqo=@AWh-r_mf12YzhbRU?hEJU@>J>wv$o>v0l2#5W;7$jIX|0eF-m)Glnqc!!xb#(}k|@O@kFCQsR7AQG-}=<Nx_`VU*Wr={ZnaOY*)YpDAdPT5WV>(St8JcDbe&J#zod_SKbl*K=DA3&ghayP4|xJVxeatRGa=oPVK_K(A%$?IRWmJ+@MDFuxnfNvs)M$GuhYOL6G`Ja29`ZlDJ~&*^Gkr#>W~e2ns8?K=qSb>%FkgMN1Xc%RRRbj2Ae|MA4eOISZopk=@>=xl#&HgCTPJVI89avR0NGq$N6-OX%HV03dD4#X^Z}D)8C8q_Y(cROn;xkzq$DE`^vjd5HT4`<ewGd-yv8cbFkkIOp)(#Z+tw#Y>E>ccLt}_4)Odf2u@=%(*c4$>%EG}1tXmIL5yvLMg^puD5T4{Ihp2KeY!tvzYS11=b!8BU2(1>+!ywzDm5#K#xSl6iqY=M`jE6Ur@6>a#<aoWVG2qyK1~zj^C9gzH0Ak=yHJ3aRf6qK-2!9UUh0?Y;=-z(P~<Yz6M=((=~+}u7izBA@%1q#SDq3OyFM=~6{wI3dfUd+@{Y48R=Y(>_uS@0mM|m8!ZCYyJ-=Byd&{S5tr(Hoie`dWBf+dhT{94!>=x{rBd}x!v-zGn!bs7+4LA}J6}$Qt`bsfoP0<>`UO(gH2}1Z<kXjWb1%!~X^aJ6in4<`D?%>YXn_2+iHpZsf$1@6b+1$La`+Pj;t}Dt9Q6l+zugE)w2Od;iN?5X0Ne_({i8VVWO4Wd~ajK=ZZYl?#Y_vh+D7U0yq?R-!E_L6NNmgtCTEk-<L@7m0=c%DYy}*nIKsF$4=gs~5@0@raeCLC&PWSI*7k1$PjXQ(Wt=9J0rt$Bkvu5KnkKGwGtz5p+RRX-Uqo!(5&i@UBV@6@z1Zo#UX8b1xn@>zOpZIe~LQ802=LBC8m2S3fqnoTpfZV6bLvBSA5F<T0=AKUI3<Dt%=A#1+aTKFyl}Q3jBVZB|Ka-6HpNXt0WRrCLvlE?!AGj3$Hc^odQSobgFAeE@?0`-nQ4Tt(Kx;dQvOa9baEeKes)v|YB;*wiO%X~FtNnR{>ylhTANb5JCW&UvyJ^}qc)tNQ;?~*V+#f*z8BC%LOn9~rQ@A0!ivl#HsVe0avaPl{adMxQz^qp?eA5*V&^m$8Ta7ow=IGKcwhZDArpcW`cN**7daEE)8`%t#TR%^*!2k$f=`VqW=Uz#m1~<<^ZEN8pHb(RS8{2&z&aMy3Z5A_(X%!mO1<|dhCk;&lsLST|S2?mlKBxRFG`Y8L-1J_)asQh)y+3>7>P`O~!HbBMJS~WpH0`$KjJs(s56#mcg?wm8bfFJQo;d)_&xpcPs+fi;oj_A)1f*RB1$g^4+a{lp;ws61(x+5AeDqRnc48@}`BCweVxLRN%|H8hS3OxfvQrp)9C{F^rhvKPCKtuiJ|&FG0lp`Wy|vv5vjYrsM9zJ5bksK982gos6m-K#4eNpogc+zx>pj0>nbmdeWmyFB(QtSV65t<T)zYR3r`Zv+#80iAX@*q;M`Oa7^ytdm$;F|}=|GCC6b%gokGGeGLnUxbySg|@?H1*hk@srr3aY1!^SXu<K+y~ydUY$hd{jhM79b5Q*ypz_q!B2m91FH%N^_K^p9#&b>H-|OH@gf08dde&r(SU0Tb?QI7S5_#FMdPxX-nc~d<P^+-Ufy*NQS{%C3q{uO&=VR5+-@MRQCeVdsCG<OTBb;QR>N9)VUcF{EXo(s7WEIsx?elO`}4Asdjyv^#jDNb2_ku0Pgya%MJ(6LR)V%9C6#;4v-wNN@n`g127|-WX5=7kQo?6T#qgwa@>agS?IDD!A4v3-KwFNu;u9yqU69m(o&jUQW&~4>{m8|^cVS{eN_hCe!?eTQ0<2<fm<osEGtGVs8qWT+WjNaCP{l%l^T@9%q_a8Qr#Z57Dnn@I>jH`+rFc_dR<)&j<)`&_QLw<QDgvTN2Q2yP>t0mPGg&aVu%U~zskN;jgPx%;%UorKIaub`eObmzKyozIkYhp2gD1vDid;O1q{GqK#!ywoL@hvwc<Upo}luFEAbvv@urC>*n2WGhKKi%ki`1t!K@|#8;|Z=n!}8B2Ae6Hq@dw{{NMid;n)7<qi_D?4`2E2|M~5I#r2d|-~X$RfA$W(YBpVQmSstHtrXs_8<p`@L@}3Psfq<<7L5P2{#0DlAjeKtrrFr)%t}AkbIW)Fm(>wtQ?ANl)w8ruNsPmBZoS9fHAniYT8fsyjI99(HQD!@!XT@L0<cF-@C<If>hK0#<@4GrH(uN*6o~87eb|K~FWk#$I$GM&{Mvl)6(}$b5958#)Rm$$2ff9GQA+LsL)#2o{-Q2b*=+1?Z9D^cmEg~X=H_SomV?7-g@Q9_<)a!ih%zp#EYJ-~uv8T0n+4FgT}XS9e9uaSVlc@ea7hHW`#i$xuAkyRXY0Mq=argoEEU6V1KsUIqAV~AR>V#8q%~-bxNK9SwEVVEnnyI7&@pX2D8C=`MU4S|DQs#~?lHL_Z6Ay^vxezAub8Vy6L-Sp9caxuz1pdYL#GS00$bL2WM(?zBpVyrbYM<7cf>j6YJkO@3)xsmMivaZJcDW@ED$93H&mgDomEv{W>!=MExM~JYkny`Rv+rY4U*Ws=<vZ;ck`v9wc+^jAOunZxUmQ^?5w!03Tjvh^KzZUW$3^7>Wv*lj`++33NiUPe&6-4N=IS8t3LNE>-wqY1JaAdiW-40D`mDCrO2K%XDCx-pwDz(0@vh#xrZ5kH4U%NmtVOn^0%zEu5^la@(yj`mVSvVvO^1|>RROk@CLAtc86l?OOWfBoEpE8KVMIYyWHApnj4WxqLLPY>fsDKzR|MM$H&cmf_fdZDLa9=={YyK72qMAwlgH3J^zW<Ui|Ea2xz4Qk}QQ8z^Vu5^!=j~@b(YwkY#zv*Jw&^EAAj>3@5j~kF8Kx4LsOV&<v#k+*t(@<14zVk~TJ8x55z`OiJ-9-GDO~_FVE*u}iR1`t}2oBZP{xl)8NX>_=I%Cgz~Dk9Z1b<3KITh`?Ylm;gd9PJ)?8Kgv!p<J9~-G#VF`*P6V~=O5*8jBAzH<*g1%N<mzJ-JNagu%whz$0faG6?WDZ=5PeNGTnp|SSNdJv{V<Cy++ax*>m9x=s3hc9@<$ET$-rw>jl3xXjAMb$8`RfOA3T8_-E{W?uG4(Tid+)fJfI((iraB?Xy`}K*@!abPp{EQt}qOTV447?w$2Je(y7l-8)Acy(gP~dH+Do8y4e`>N{qj<s-Dh6T9`)#iuXz^Z{79E|eVL`+$6r-!$7-ca=gz0$j^3MYzgPcZayNW@Z@tsOpmeXnzkU4%^z_JHUUAH}Jt&U86lHv-Nc6VuyJ?A_ypIv$;CJE8--M0b-I<$pu)P=`$MNsZak{0js7()ujl56e6ZA+YyKFy}|dHzDPABlheydaxkB{@;Fucs!v?S#?7m&JrWyfPy2gudpe%9r!k0($)OQLKo);;U#NSqVIcLx6t@IMAlupnR-^0o_`1wSF(1uR*t08x71fSNxZ*>#u5lM_Tk2)3oh8oV`Z^2QqJh3Ow^$i_M!WXUri&m8a67ifxSYDJxLPp}G!0#ffllbc)qs7#U0-ftpv#UGZSPiDz4Jf1uV;YNnPX#$i--9=-5RFu#ejqF>dq$fXc1o|V%#<?0S@c}qruUBl<e5tdx{-WT&Z9-0Lx-aO&3Qh3gMT<;+!$54jE2EV#%##NNM_$yeYR*K?t-S6r0sX0)%J1ID6C$2bqkfaDj25Z)4aZx*6$o;$DW4A!QYubH9^?7!yTC6!S$%v<rt{5&y*Nqb@27_}+Fnc{3j42|Nn#byJMxF1(hvvh95_VJY@<jiYmQdQ!Llh2tqQU?Qa}p%azbl@dL^kA|jE-q<%g(Kkg}q_UaLxO@=kk=KNRzBu9o5vhGRJqPiQ$X#1MlNFS)b{BSPacoo7(XtD^bd>torA`E$O;s3RNCZX4qe4_9Y;g)sB4Hf@Mi!Y}4O28)S<l-TFdc}@IcGz*&AVH%c^CF=XHbsA){Xzkou>8HRXs=)UJT*gh!|}e1LyfD`8(Kx%Xb?N)x9SG5o+N}Q;(t#a>}0zJt@sn=EPDuC>!;$oo?x>_Euqp?JC@5EqvQw#L+66Y!_!AI{+Y+p%wdQeC56`g6h&^yX*}qBZoLbTnO4*q2p`>?rIaTLNnU{tlqNu*H}iEY4_72DI&FS6%}FPx}IytqCnOyaAd%;UYnPRk>^62bX_IL#LGMeXL?iad8^bhOY7?vq=seZ<l=G%S%AYa&h->g*LaatbmW%;d+f@^NgE8-63$wQw5_Xy2@I$WnOXB%v>hGX*wU6tc&~EPw7inQL`MSF)b(ju5wVRVLpv{6L41=^^MlUq-<e;$)am$`P^FK5;Keh;As9g@j#s4+QieB5Pc)HmcMSg_)gGZhodM-X!*hTT<}XK+!Qzo~#0y;k-o_)1vr#B__ZIN06){f_jv-T#TT~-cHLPqt`leV_ma=7T%kd-VHr&*Sz_#E_3YClS8tW-pdUeD2LA^wH)Bz%G9Z}m~r;u!Kgm+PRaKq%uv!KGqnMyTHap>`0O7Ur95NzhCofE4-D_q)EI|QfS7%w<Z+uH@k=$fz?UmgY68tb2wpQF&V{oN0~djCfseCPg8@Biw<cR&2)2Y+$@FFyG8{cqm?-iPme_+C#%sk#9l#ui|cLYT!sx&#6N88@ulEs!rhp1}2)L`-p7w>varx}wo@p23Tfq6dAR8f^<<-IRbU=2!7zN+qXx)8=4SdA9+R&L;tiJl(Ml7hHu4{*sh3w&9I)AX%#mOO!YbOHG-uo|RZ@xCsO<B2jnEE0-2Hgr@V-*@DWX8B^k(Icpr{$*P3>ES-W<B#@ed?auPaGh=~Ny)9=^!<eiJUMkOVF}XO|9)=sr4uT_yaODdgL!)zTD4^)V6=h|?gigVTXni&WU!;wi)Kd=v)+HDCb?i!Dii5a;{jw9xq>iwoJDsgM0HP0O^9=8wE+7;tsjWd0ixg_z(bRe*<hpZ7lk1U_o?>rm`=az#+~j=Z#AbUy^*~OdkZw^n;hfL`lJsPcPA1Yh542=vljI<h+iY8WRWsg)A|7BkxEfK)4>Nd_sfoaYY)$@0zy8L<pMC4$Z+`diJKucxiy!~~x9=cZd-%0q|Nh?hKl<ihfAZcpKK}M!e*C?EqPO3=_wo1s?&073!^iLc?%~(o{^VD`dH7%c;gk1%gx@~;!GHef2Y>g;+wVR6?pG+P;eUPS<DdQT;d{UR<m*4JN|OHXfAh<SZ-4*c5C7reFMdFv|NGkyzy9lw|M>@>{OVT^|MF`OfAJf9`@y&Ign&04J$V1;GdM5s;NDlgz5D+-h5L%;@QThdT^rs1IlRd3zYB}8_u%~>C*B0U8H<eT$U|0(MQc~QlKG&$bFpsO_7ZR=q_iMaJW@>eir2n$xefx}v%v*12SA)yh)n#_W=nQ$E(3&^*rWr6P4hldf;vFpCkUo^d^1S;sf~Kpj&O^rwP5SYSNr)WOlSd9xdJ?8F}2ZttmU1l5P9kbhk`92(Y3PzfMh>Pg(ZC;0ou~T$QxSH%$h540=OXa^0JvRRA5``E(4dSnX;MJf=OkFwg-5~9YlIJj+x$zSK>Y>#K|zffmfr$0rTs}voUxi(z}x^*`RZJ^px}CmB;SvLR<0h?iG6R+}$f+Qh829FL7ejjX9xXib!{H$IoZ58$*SmoaStXCzDgUOwa#f<1;&uX7Vm%cf5;g23#M@c^B`#i59-ln<U72?qZcGd5mPxgS&8Y`~Yn%>Ke#sik4k?k15TOB%i>f!g*%63>^mghj;(SAO7}7zyH^_AO8FYKwRbbM}PCvkG}E0AAbFpcXQ0c;2n(a{|cGHgL~fvBKF|@cjLS2)%|~*06%!}{!dOkD%<p{Gi{Ae)Egzc%d~Mo8=wt3;W!D2u}~!XonX)DNatt{7ZcYEh==RF86Jkr;0TyZ%H2Qw^jn|2_mz*|`O4jXID?3z&|bXS)#DMs|DehkB$j<83Fv!)3vEyv6iF|WoQR_Lm15$^qU3;$Tal{otl9Y_SciO*uS>SWBTDM#j?O?)(tLLqGFoV1yJ~Okv}E;GLMltAJ6hZ?^41~(3NzU;#UVT%Y;u|%#i2?qSWLwPgzQMX0V>(?j!MK$f`lY6l1#eIriu{Y%sAllfmExINt-X1NXd;NE32=(KfQ2)%J`CK&Ax%6e6NI_Qy<X*g)}znFU*Z^`NSmqhjgjU*>0c*V#cUm-;KGP>1Ow?b#1)$T|($2ak*qHR~5mZ-NW2Z8W1LXf~@4mfetebxz0dNP*+VuUb^&L-(%lZtr|(H)q|0boWD{Bx?({+sis(xY|F~7Vy8TDBwwyN?5T>GIK&CdXqi1-cEpLQQ_AJ-xolW96NHAInop)J$h=JRmP#4HI#L3%cS<K5Hb=k$C?Ws_Dvz~ew!BlC7IlgSwXHFT^lgi@RSbF+dzoprl8I0>r;_%lk~790n<U04l8H(-EuA&qI^9~InbrvvqA?&-8d1c+%GK3{Y)92RsAmv{1~1Nh$&=k&t3t+`lf}8w6_T)W;zpfKo&g)V*;HCKYKro*4cAYJ1VF4;?~D?$t=oh^-}j5JOd<!qfc;Ep+Bn${`1+C}#T0&~Odayc$J84gEd<fD=#S+a&kKOoHRshU&WnP}FE{5K2#Hi-=i2&8L15+a5N}iK@?A;hT&Zu8q)v4oL#v3osB)}^QYpCk7VXcf=B|9xS@Rc-=XEOT5w!&72%LDJx-UuEl9@U>h<9!8l;)f6V3rG+o5j82@&=ju>00@oa^p&O`}(2kCGA_6+nrY3rRx_v@Ua5Dbf5Y&9TS3mMLGgFQ|V!ty$QF%C>MTe&`l|{ev&bbp7s6n9&%N4qGZViD5U0eakfu4942@#)EBPbdi}<$H<7=xDMB;*Vu6Ezo`WWN+M!9z<W;m^4YW7rv(z^qZ&nV%;)3Zl_ucWjh?U9j!Ue^<t&8CuKxQYPb4`O{oY_g%mS4(u$OryWp{G0_Tt@wL!31?yV&_h0JL1I!>vIv}Ck<doosOn!6oL%4b`k=bA8}F^J61v>_?l;DW@x@_E|#izxV#Z{ggJ8a=orG+GKsJ&b}+^1<4#nIzi8tTCxKJs+=$wH&w6t@cQB2PFv*q23=6Xo04tPZ%9xV?@)u0;RFB%L2F+zzMs-1L<s))67;!9g-4CDgrpqK@>YS9N&6|!~t13E3NuidDf?W|JEEc{QYyq05{C3a_GrA6R8M^kmnD@nuV$I5=qH62aC|%he9#RZNu{=eUY6>ST%x&TN>4wQH$=^uG(I4qo-F?E=G8d!NBmvB;boQg|yF0>$0DZGbQ`<yjg``Ku)h9J`u(V5f3y>Il&CXOEs<Ny~IfOY7#Ns^Mx9A_Z4HHme1fDVoQjr>(fe~c_i!Io~j*O6bjV|)EyJHq)6q3|UEzj2XqO5qM9D>)#D4U}A;(A=uLCm*Dh}W+{hK1|LQGX6wtbuJ}x;;!LMxBr!9LEeL4NZAY_g*2E<A-(zHZ@H8qm(gBt6A2@%$*>EwG((kx|Zm;C9o+o{&YTxD1B*&w3pokaI%0KhjEqjLL&680BCb`TItd<naW*4VkPeO@TT9bQpNy#rm+H}Wl87e2odExTL6iT3rda!NEL{zs}sQ8p5rK?)o0v^hK#Nbkc!?^05`QAP8ZAZ*PXf}hP2Ogb#|QSC>Zh6@<K|WjIv+^9LijKH}wL$$I`i|Zfi;>)F663m``RIBzmJnm8h@-anfpmPlqZrK{IZo)Oz>|$LtDgfVBuMzR`hSJLY>O&=n(2EvlUhoU;7XOl=G2btQm2lLjDup|GHsjWbZ0AW9yjON#cEC{nzAcb<*-L_-;;Y0G<0nEDLdn(irLfWPR>fY8#F&_EN40A2SG6?CX_1@0&k<6|n^!F){TX3Pf8feY$T15;+dOfcYMg<3z#@>##n9J2|=Gmr~W8zEOn`zENoe2NaQJ=A&^CM4tYipM!9N+4LKq*{WIMVa;+g~1F?YrgtA<>$tXPluq`WoTFgEs5TtaL#!MV!W1tZoh>{Q(z&$)bUKi_{5`&{7_6IGd4Vfua?%Jd=!+FL*fn~{cxIVx(&cDI2B2pbk@cXpWNe9H6SbwO!qAYCDAKLRR<Zr$hT6m5XpBE6ybPK`J!q0OtWk740)YJ(LtkQS*c9vom@GPz~|z2DK1DtFDKbyKh2k?q~>xo0ThiXeZsV@3G#9I{_;z~%@<z2_7XBcS8i%a>sy=EmqnsmAQONpDMHazF*Eg9EcYl6Eycb7hCB<#F=7mSk$mW}NrVExCn$qgGx1hMD4C04f10myiz*g_<v`Trd1%SImtVd1QgHo0LS{eX_stjn&CWz+cqef*9xT3Dx(4megL>z~r#Ego@yU5<!IGtU<EBb}h$Zm2RM~pPu{4VCXcA?pMnw$)RZuwv<Wn;)0hPiTaTOO|q?9l#v5r_$DUVskJ^s{`$^Tz=ucOS?U(rHD7p71p`>GhH=86?+Ro7XKww78;Fsq<_S}3Rme3C*8?UcW)te)Ot=g^H8H{o{ZMT%)!R!VnVslvr=Lyi;-@ex;w^0e4GE8zlrVju+vsn%Sex_%1>F<ifvr0IM{>MR(GWS-8}D}!Kho=g$!(dX`_D&n-_Ws^>sZn>4F6;l{7Zq-dGBAM2v`IA1i0=Sp5wy3No#tKyQFDop!w`Rn3*Ml-$9hM%&_388>X9OZME?s7V>1T?Bl>|@Mvm}w!RGsYEV!*rQDb<n5vUD;Z<2dBpGrOlH&*%tEqe&m`Atou5j-@u>hXjq2{fh!=xj{1nI7P+}F!;#Z>Mb1(bVwfy3sj4A2>`b#@NvN9L6L^htV{@d$A!2M6;qTAz|ZNNnh>{E(V=FwH0;~i98gGBUe{4C98~~yz+M9}YLPB|9z=W0LNo__JyXR9WDU9S-bsKl@-}VdI3~!!n9Sy5cI6?Z<>?j915Qh*$9{W^W0&PLBi^%mROc2L)%m%wyPzjqHzMrAYHPP#Lk3agR~PU*dL(`^6g-r!e^5^FF`Ec5_`oYvHBrOTY=H)5>{32WUcZZtX;=uch~{U2t5BF#VH=7dkq>2qGka}<Qi=7CqIiGAqzn6WylYNp53UJ;Hw6{~+QDx<t-V43QOE{3V3j~kORqR$iO@W>#^j@{*xJxhBU+^AAQMf~I!TTmV|^(KpAf3)T4`t)->cXPtGvYUbD2sgCs|mwkB?>1w8d+Qz02Mj4!9<IVe`c>TNb8JVhlxOs$eASXE{bm!KMAuu|KPTl%h+4ay7NfS3*$)<bYH*SXhvTC4z*%hV-k}Cgp0w3QVeYmton_ORd5v+jOC>69Fh3RI@wgZED4ZkXNfIT%e1#g3=DqhyuIL+z}NhYTE}EIDn$*f$(lx2W1#j8-}tYgW+Q_Eoc_06RI5`-c8Y*aL#-Xhf@hBqeK<pkf3)Bp%O}xAd0aX(AhJWc}}p-E?u11kK%Eo{mf+`YIHP5VB`MZ>aExUC_jKhf!=kBqC8tzq(me@cOjar(0LeVbZZ2Ky7JsWRe3tA3F1(=%BpBv4(1E>Q=#;?A^<;__hSr5<W9jMhzPQ3z;Pl7%sghb3ldb*aXD`W_9jo9@|ikvf_MB-`yBNs*{4-PXo9KrE^GWEcC1+!+|mczV_Snprf)KV!{+uKYbm*@Dh@Bpg~KAiQURu%ogIM1DslfK2r<efs=+u^6zn=4L?0zF-oK@bx~`AGv1|DTZq;D|b0Faz>}8MX{t47GF}Ol$bw%7@#UyOqoMS50++N962H%vg2n1a^tB7%5U5*z~+!Ra&S<ngcB-mD8z#Kaz8-o3zR!+fv4K~{VTy1+1Zd38G4+J3NPn60}Kq-jZGgGXw1kA-+GMLf<KmT#>r3df-4wE^<c?q`e$X!lfh+l}4S8m**vyEUzTHo-ljUL>4dkT82fthC>sOQhZmbw291tb3?m+yqeXU}^5`~QqH`QYB)!UyH#)6xZ4A=XP%fGyFb?jT8+RkU=`1E_g=|L1VZ<9KegSg;T*@ZjD*(K(SHPBDZ5ip}`+i0Z$l|Lz`Kd+`3>PrYM6gfT`~n;j8~hTWI~{MZ}Jp^d!<_x=+YNvef~@z;#>_Je!>6Pmd9S9T|XfS}m}P8j}}CXc3F1Fd0fy`A><6Ygeij_&`<1ZV9xazlig1!0~rmcs|{{|wa9_`$vRcp<$$;>3*?z5NIGzSFOO^$F^ajhq(a`|lK&7{SVqKKx|}_F1^15@XVT=N$m-uVOFo?Sp%N54ZtQ>uv8Pj$WTZp&!REW-dl95m0A*$8cT)&>y}_b?(1AD`u!<aAOQ3WROXQ13^EVs|b2P4_GY_S>Ow1A^?=>L-a6o6+tEt*n&tc>g;-}n#R<>>-~plk1|}}#g+VP0E)|Y*L#K9x$Awt*cVvD#9H`$Rn6R8++}yYjg1YD{_ZUP$1jvrQ;IOok>Uo|`To}LdZ)()p+58Yx2e6TcLgq~H-&znUl~Rj^{UW&;LW;0Ws0PNluYh%m;;z}y3XMKyRi@#|4U!`l5lOqVW`pF7KwW;au!bW?sk)_`YmUMS&!Pg29$^QR_={I{N;mtKc4Qd)@<pFy1Q9r5voRS`*5$t%QtTSd2s9cYp>qA-JTF%(rF?1xO2s>S0m!!@th$F!M2?oE-3XOK8`{$N>TG-Ay?&H;e2k>dsUJnzoG=Ym22A(ZdF53tu@GH^Ksc}oYEb{#wiy;*~?tHYfskGx@(UtMoPXcd&kLxYrAu`tAqWfrndv!rq|MlF6*V(I0^k@Zxe@l0arzKjoDQ(qKIN|imPb(x^MKb%kj(`FKS;bziCxmDzdoXsez6<JS{Lrcy?IxU3PZm3|1$#WFV3-O2_#rJF(M?BVAu5O1LQcX*o<7U{>!p!MT_mz(_C?DdpQLy!4b47d-QFF`U_@R*j#@I0rPE=kd6WInwcaF*bzQ5>0^e99vVT1W4`;E+Ylsb8QTdO2&tHC3veqIV=8MzYVNd-L4~2w~nBc>4^HR7tB~ExhmKd%V$ggWh1XtYI*hr=*oajcWa|%N(}2eg|A2}84bgOml-)IYEfuqC!PwarX-%6-$c|Ra|GfzYqs~t$zH?1Aj@A#XJUKDbV_U$UdUr*OS~NCAXfspaLn{6u7fiUml#ZF2DC#EzU|79ve+etv6Wp<4^;J`@PL(XeW1e(CC?d@w*6#lZjT$CA~J`%*pKE23KbWc^cUMLUlXBjB^77h;_-0lHt#(LSwQDTjMqUQI3lCU8=gozL##}IbyBKo18r#o=+5dFS$}Frp{hA@oX4W-i@A?Txl->eIJ@L<)c%`KVN^HF^QSa&jQ&wl94+gRE9FtmmE;Q!k=0|u(gS7X@aS%#6<1HX*&L60O<b?Y^t|X07louT^R<s|Eegd`QFwGa0SIY|!Epuyx<pgzT}}5%zU6eM4ZP+A`7g+O-GH31f%3fUq5F<XwZj3NRGaVduM8_e_E60H30Ua2^jUF(+RVtsv`00$8YF9gF@&i)={%!vp>(I=Z)`Am^z{L!E+*FuH|dhTK!k7r04GrLS{QhE$=#u9bC5U=k85L2?HG?4t7h>WUu~s`?yRm>Qr609l^E~71ntMFHj2ixu73lQrDk}#4RUg3c_BXg3xOpw2P`<|D6ljXk8UgEyPInvrV)4&H$Z^%6{K7E##8B$U>PZ40NkoUY?sf{ZwMKLjG_aKXOpXCQ4`cHK_Ge{)0079{j}c|45&vDGT(g~iz~+q-0NGrRD2>#>Oc@$Xx3`iB85I9=4$O??ZlHfxwtYLMtNCY<aZco3?YP&Hp*ytNgHzZZ$lmLs~O?t`XLG+6qlAj!E=90p@4L-A*Tr4P@Y*1MLZiDc--*(U4a8B8=xZ^vtEz3W~^a7?lOgb6y&Y8$gmc+#VlqCF_nG2(v~@I2e|{H4W)EKNZ1AagRrxPR>n#viHrTYMaf-I1}YU%n~@5190iP>!$7RtLa-wd%iq%2rmCQi7pud?&wA7t$V{+L@QR^BMa9S{fg{!tqc9eA_M1eSAfQ}9frvC1Kbz$P)Yp-H<M~5C(yB7IrZTcNnrr_T!RYK%',
 'diagnostics_snapshot.py': 'c-qZe>u==Nb^orv;zmZn;d(@^w%NoOJ%G*b8rjB=^?F^SQ3aYqa%O08$O#{k#_9kq{2}Ov0$sF05d=oPF$~v9gC=#0qN@-6ASeHa`j_-~&b^nsBxgplV;B{%EAl?hJ@=e*&+}57o16Eu&?`dWiG1qCabKjPQ5+?q2qJHsq<Inf;>V93i&2(NG04xn(^zC>(ic(U$7K*DV=?oxBJ#L7cQ!XSH=^k*%?jaVV|3-=22b|WI1c?HN|Riu0r%4~DZ;ETf^g)OaS=p*p(p$KiJTVzy&{^1dXj$c3;O?qGzs;TNimJ}bCgz3{40)z&NM7M%=UE8`+1tkHD+Eh!3e#?W4!7?v)C&}X*SjG`8?N;w7!1+AexP$INaFSz+MDs8n^_$N09sC)XNf3Dz^t40zXAImk;!lg+-YqjEobfUXb$~hd~tKS<wp<KP5WvTV*lYdd0GBo*SO|;j9q7H{X7kWhsdG=U!Zf{A>#^r~Y~ojez6LW?>TaM%Fp6l6cc_F%ah%ys+#xPUnmEe&XdDibf(y3nrrzo<(_)_w3FN50AJdOHXsLuis9iVj}0IK&-o0+DTqu{b>yk><~#g9g^g*tVW3l(GX@3jh-Cc7dr+93P&33pg)@RTAdxPm_LNECY-@0f@a&D8Hz78H#*EPf7UbRTyC?)Aw^F@OYsMX#U0LzFz-=gUyPzK4m5KlK|t~-b>7D2lkuZBdr5do`0rZ^=4lq;Bw`RiIS+6S-{ppinhcSU%J(JkKK4{(UheUc-nSxXi!5wFbNJT;vI9V}gIEU?iDQJRA)v+#^z_b;!}&lSilCg%^4<y26N@H!nT4*G`%$$2su$;>O`MJLD9H;i@k2cj?8ysg7Hq|~MTmiddzNyc5-Y6Qm*W(Q4)B>mnrRVqQmfqmqQMJECTAQLJh^8(EXiR{%JEhnjX@g8ET|3ez|G6))XV0*v;>+-dfGWu;YpYjbX1Jfhrwc7SaCYeEqaKOlhpTEsX9EuGQcSALp$?<Bf1;(SKzTP+9ojXS=@v(x4$neo@WU!3F>|v(To>UW7ry4N2Usp%XQBFxi$3iD7OZ}DiiO-i(+pWhd|cCq>!3_G`ai_*b@Kp3xVB<VsdryQ6$EZmk9sr;xmt+hr={I7SpSXUqrj&SWd{xxyU?GTwOf##h1^nE`CnWzZN+f#A%uxL#nJrO$ty#;&`~VqcE>%sQV5Ly1G%Ib~ww@aTeywXt{<%de#c|v@}#j>LvnA?z&0h19kB=_=;IAuHIlN0Hf7vw&tOq%H^q@NO@?5u{X=Zz*T*UYIO3J&Fewr#a3m_9kRnwo|38cik@PFS6D~3?G$O@#Tu^CBrwZiPTbT4CL{>aicNgx&n4^qbUKT}A`CD=&Xjwv+TCIj<!-%iFzU8CK>j2Q%D6MNYS&ts+^BtHrR{3Bz^ZO0+Ut%3aa*GK2Dvfjd8sH?9~`;3%ZJXu)IEjY<`NC*I5n9$FgV$-K{9o*XOMRw3{qqMC`zIn8$$;UTihe|+GuYxcL4b*VXxGCU;L?UEKrlxE8N%v)|xArSakwyO~gS>Xai80#kmWu<3eSYtSIENOzd0Yd$(y}mUU{VNNMg>BFI1ht%9%v{?Mdr0l;bO?0V{*^{{j;8e8?Q><aKs4Val;jgE5Gm9)^(CACg`YAUr!m^QQ`J^#*E$}#@KZ?7&sDU^RczIyRlB9h6~i%*MPsR<|P<;R#qEh4}~Xw2;LSBccM(bbFpN#r_?c=+_5@B#a8MSS_OAh^Hq75rrESQ5?JF@%5Y!yae;BsvLuFqUwKVb=OWH7R3;u;4NHA7&>YMuDU5>$y|tCUH7}^DN$tl7PI!VHSm>cn+jtXhA!{J|%@32t_!;(DvFZXvKncu$PR(-YbS<*Kz17>>RarPb1kFCePqT%b|N5W>8Q$PSR?x4(x_*08&{GeUR@{_*gU=N0WGLGv9%1?=Ix-4*o&-jBP;me!uG{W?qyx{T9}kc^rnb9u7wDwsZR~PQx8>>y`!u_2s-`YnNkKMsZLvi9L=cd(0*%F$#ca7*0MvHpoPZd%srPc<t#(%qa_Bd4S5w)t>&;OQMlf*>!lNtflT_BiVIfJUUuS9U<AZ!nTHIGX$KhxX)qjVAWyD2cl&!C5VO6zJ8jKHp{}Fl)y)#-`889S>&Dbw3Jk6Pa~pcTDCKe(_zoLCAVTX!XO&b*nTUR+A!pZ1Vzqh3Ir4pPJ-dbl8WJR%9@i2S5j6HfOcz)Aw6h|@9b9t4rQ;T5KK7@IBKqYO$7Pq!S*{(UK7a}dM1L4_TtxN<)Rgrzey(I<nrURscI~H{mw|GBD^5>aWr6T0q?eU*!es)AkjAZY&_^!kWRgbq>79i{0P-tM1}M~VE~IVr>99#%-s6*C#4rh#k^{1uAzHJu{Y-7{=~~%pHdq1#?g^q@X@!l4(OQCe#02IR?xPXyC7oTsY@i+&-ui=bN73^6)aZ0HerOx7%GScdA<+(Yiv);__C%3C7KYcF}x}F7&^N#wc6{DB0N~#8oX|4(OEaEnbmUkqAd!Cnwdv@jB1Hi&8$q93w3i`7qG%#F_7TL$VZrK=p9<55A3BNf(R4V!;nMP00C!Z3WWhE{i3&0$-t5Az+a`5lZ;Ipi|9di%egdEoIb{CUAvK7;dnvDHLJEiXjx3zr=gx(?0lLG=)*Y4W6aQ(pJIJvrNApww&Y|=NwF7I5H9ybI+=j($KG@pc%<Q3-AfGYW+VXrIB!^nW<bXfr2x{F$oc5TTFNrXSW7`sy0@{8G9S<sV>U<K_>q@u1Z`@!tf#d$fuAhoEIFp7q2yRP=rbDHb|jX~l-kQ~^cf;(ldsBY7P*Dt+?7X&Jb*fB+f$j`PLf9ws1}xL{$aIRt)49#rY8?$KE0si%ll%dlQoi*8W;zslv$GNK(}Tb)<KH7lraD~o!)9n10|#~j^X`4ofrUAT8AX7cId}p!pTV$awU{ks$yLVuaG)-$&#WI2cC_p5f4Ox%T~hWoO=#W4C5-FNtyl55W5W_oq?@u0W?O~jVMZSWxy&;vXv08i8wHn9fyqS*r*Qmwk%52v8IB`!4jvF<&<XN3hGP!O3|1pd%Xs~mh)+eB-tiLO)lSLo%M2Y-Fk*vyUAjv7L+R`Q0+J0mOSjO6{vBUE}|yxBgMQ<Z;%}`S5s;Zr_nDdxXYCUsCIKG2UN&~lExuhMniFcdaO+wmQbsqWm`1>R!RVX&gGNZt|YG{mF(zi-BEk<)pVWqwHF3(nYaby>od6Q5W&mW4Va`KO>y>-QKdlt?F4S&1R{^X`^D$vAE%ekfuM4rXK4K-5+2HJsB(Gv1R>$+HhFz`n~Sr{=RQaKzeyMXLVO*siv*5#B!bcm-C3n_5}O>1Z^K#sy6$q*DdUEQi`7}>8vlB!PpxF^#vZn967XfB;5Y7Me;a<bt&-YdQINGVN-avRmla#6>>J=r@86Nr<w>Btxtw#T4~aCl=?M}(QAqiM!&>b3z5~@z%x582Y!CM6f23AX_1ip&HV^iwPAf<}^03(7<N^v>Sp*O6n+HUuyY{wamq9~oFE8e38m7Vge1y1iuyc2IMzvcU>#Z`{1+HM(;LdkvXS-Af+nR*Yc!FWh-9<sAiSr>;OPFm@)tW?VpdWTmqo9}!c5dJPlLdnJB2Xg-JG4L=M}c@L2!h?27m)E9d><>2^lU-=$H^cLN5w9vg1WjFt2UV31>g{|Msa$&b((pzftSorCt(&Yz@c-b$BJ+(ho>G6@XjL^isa{hnL&LGW@$wFe6hDJ_im75WbI9MmXS{U?_d9;I4Al$VeWe~&JD5mVFM24G=a<xw4xd`XSEy}d*O&58VSd9(=%9XpBpa3VizT!e`mdQ@sS*;J%xp{w>@ieb$0MjJHniu{PNl5C#>?G&qbv{c7^tUc?=YfIF7C^KAa*U$c3Wc$@~LUB42Hwk`ro5KQCmalyeu#wrqn)JorO+yTfu^X=Fn8a{%K_4k<s2;j=PdhVqwxfBDNfqmpYi53)RubT;?PX9_zsri9ThWoZ8m)#|H@pO%asjr>q1PiC}(Pkl5&^Al=6i-1%l`H2edm(M(jA869WW^(7iOe(KV8-x3I4)!2x2bOqQn7lR=g*@xm=m{J1Eg+JGT~sp0m6tsWry^#uY~%OEKhba6I`Z2eF*1$Gd`6)YGW?qyJbC~fFyY6i8DLP(X>$2Is#@Q=Up?L@Ia5nijYwU|=WRe#3mvsX59Pnqp$C_GbqP*Kxg8I*zdBs&FDXV_;;)3|ypO|tozn%n_N-k&{83)wp|YGucC-8^^}S36SPCPR<g3FXt*dnmZxM9~xTVm60w{um#<{0;WwjhAZEo76H893vvQ=etNhc)-1k`RA2izDbXu(73m3$3g0irlR*i$<0;3*EkGqCme@>k-|-hT59lHV7f$pi2iG@cqJ1-9RiS83tMBa=e7w_QQsX8WTv1LUwxDa|~xEbAq)L|sDbY5j--$m>@Z|193Cc0sV-L%>_|aU4hNvL%<FCrW{kfaB=?47&h76K7*fyr=NyT=9E@Ms^g}n@myei?{Cgxw4HG?Y|mD?`<yITvH3;HFdHC7d^0~z7wV9LJ$63gk-MLtaSb+p?}P_rK9}7H-@MVhJwf^i!$g$mF+Bip94oY+vKGVL6t;|Nnfe6jUvz0i~k~hMw>6mvt^*l3Mil|z3fl^@LR8imLN8)Ql)CgDmx`%wZ^Jm!nh!(=Z8<9J{IzXQqEc_BQC!^dZ6H`B%T83Pf8wx-*bRW#DuG6%({A3n4CmenovZf`TclaOi-8=K<K9upT8-^1SW;E<#78WD$z-r6#yzu$RtS7q1z?ffKk~LWdQmzOrYyId@+px)pz-FWgmje{|0SY1lbPf1d7Co=;DdHOBl5+2|@78Zpg%QRJx8Y|7A+NL%=~?#H3x8cnRKZ6nG>N(_vb)mfg8cfPbe&48P?lr#Qogro29diRKjXdh`PZs3n;|>gD=mTs|e5{;Q(t@DhUI%m2QB!GH0e$!@Dr8e9MOpM#G$`4xfbSR+r<w6L@?-SwbVsg~+f`6CEPdT|3N3Q<dniDwh-rVtY5ITyXwW`tjIgWN7)+RnxX`zc(MRfq2zb6CU^%#a{>^HM`4<I7)S>tOAKxB|3xHzY!=eN`z-PHQs*qTY<c-h(PjiJ7(L*%*2$RMW*jOo>CEDEAMze&KC%!90d*0huN-RqC!PC$a28vwk6cpvKJ~-d;liI!wEo3d95HXH*A9b(&6MO{QkJP2AH}En2)*L(Ycb@eXe6@($WQ?*@zyn<G_ob#R52RL1leN~b4Z^8Ke7UxuT|)lHB2RRv_a#0@*9A<!L5&MxXYE?Y1%$4)VwDQj$~C(2z~GuBO%kD2EVY#bH7DXFKf!=gh!I{JY#nTS!2y)kEU&KC=#Y|70t%RifYa7j+haF&DY%){x1;>|pIoz17^EEsW5jH>3E!oGq0VU5Ajp?ai3O7|MD-c?VHvfCB^B~u>Dp>L$ct8@{BZvuQ}RH}MRV;_7IR$U`@ejLqSrP_{hpTZ(~QOcCA(a@F4YV4!O?t_P~zIN~F!v}m{b_mSOq&p|xxlP`$B(KeS^6=h+*B?4lN-9Xtr+H=3YF%yZ0BATq#H7DWl%uG7?5_yX^o9UXI&NPPhM{AKT=l(JUD;PxW3K^YxHJtgs{N*53@4@mMrXEHWZ&%TcUgNht!>JV4&)nF)a>?680G2I*Ep&o$t2A~Dg!B!91iX-8?d&@CWkDR!F~C<B3Y~2RxNh)O(7z+b<o{8Wd?_3fY1tkFpjtI-<|KOU(M@4{D}k%Az5YQ$PkP*>ASA5wdUGL>&^+DN1OG=7JPs<*WEO&Jy;cc>4I$?$KJZdb=t+cg7Au3cJn^#Dn>2858t^2(j)_?XF-b_QJ@3E_wta9TYIxQSXYhe?Wmj9Rs&kCNBxzz$`y*P76m|12@=jQx0@GBiCAylp%rnv9A-a{J_yC$zA1dmNos13o&gI=Y*TyxijdnK8rsXb1VSEmwnny%Jdy4v5sf>*D*@%l(aeREllmuEx}D5%h%1~sF5F9N?X~Ip>cfgpIrt#zOBK4xq;FKfvZgUE$SvPA-{qt{Cvb$90zb+v()t3JcWN0BWEE45m)A4~D}_YEqeG4vCV0v>)Ic2%7g^IhI$MEaB`^ve4xFijR%2lB!j#ID9IZL4sM3|oT1~EBwke+8X<y&!%eN`}r1xU41!Cyrt^Fkkt(|O=$U)k^ce6FW9=%qbgf7(Xai}(_j>M8X9yeHJ%?i36i+U3TFOPzdOS4kXr`f!_oU)SDbt1sN)^{<oCoShi_0--AeHFj~Kxz~puXT@>B^@x|EOAM+uSs2hMc=cX;b|^u^!`!H^4D(^zi}v~f>>8)DV*7y9X-kcwHswdsNHHLhniih#S|a4L}^9_5ZC&LVH={%EV#&(Bwfs)z6#8eWm2PSiF$0<dfM_b)z4P(8XenEG1Bx|oIfK4!_}@1g;!nk<$P54mhT}7f=7du`2Dd~Tj8v7W`uQh_ja9dv=4xk7Szs@Q8ca^^R&!xg&$1ea8fsG9|u?EK?BviMkd~R=iY0Np8kdV<l*BtpFEX;rIy2HU}xQX=fR_=&4K#suJ)l_vtTuaf*vRizH36MLPOR&6ljPas}se-Z(01RY~^YZG!D5uP%XZxRzwvAU(Bwy*gTw#S!>8i&K3gqd6#C@y@9TNt1lnmn~9mcS#bSQdJ;4Qe?rg(hR7cup`#vuJt%$Qq9tKvi7%oMXN;mVeWOQrz5DP7?!zZf-hA>li{4s_sH+wyL}jGuqH?N?mUWEe*w{VsQa315pIiT*iO@^pNBBO3N)RZ{MPs_nBjiPu#uYE>&q>mTl+X(4+q43I1VE;bm3hydRZ*)9=ejua<uQD{gMu_ma79WTu^HSpnkpHrKBrhtjs{)TIyVaem(rYl^{=a_SFop!t<EKaNCMX%Otj)EWcfJ=p$tXK;_!#0mAz>@1NB5onACFCRv;7jQDKn5r_J#7`dHq-qBhs8RdAPmw9*(>s|~)c9JfCAx3b~#Py9P>ru@D-55r^q@!c|hZJF${?HlonT{kY_H0)|$uSsc_o^AeZglj)}Qy0W=;$kDgTYr64Cb)Bijc9M%OyKJe%B4b6-*1ru(<J3TsgZGdU;cRw=Uz7Ei<XbMrx&2`jX!a~pXIxB3)6LNV<ts7KweJiSzB9>8cTlh1S1+4sH_;TcDZ~G29gk~sr-v3>#ug}+tJNW(rAdIK0>2!j7&npo?&6blRmdJ3WLV*)^+6SN}6%eyKybK8lanw7W~B=WZC6<iR<zw)2>VOcU^0sK4c?iZ2TX}iL1x',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ<%aYqjlHi?Rfl=EUKo!YkrmA|>l&CSaq~4OXT9QdwRlSy#0|QAgiAoY+0iaS@IeFgm9{09pV{@Cs9`-bS*h8=Ff7G9>yN7!q0tkRiNxjyxvPuGor-z4!zvJG$d+#^VdhKm@vvo2bZljCXi{|sVD7<;L*+zL%Wa-44<=Isc=iVxg7I9v9N$S0ef8NDKSq!{vTP9f=t-bg0I!=?_W_0h~z1}j<Hqb^o-{pCnmZRma+~qM;-E6bG^iEefwDmU2)-OKB^Ie(c11~B<>~tNM@j}-x<B#P!nMu8Lw_fwx9>1EG*W0*|r;A^bt^EE81~0!iQMr=eS)rQW!aJkww?>gXmS4qcS0-!yw3}`73<0FyT<f1@veDH_nqZP%sIGST8hRQ5M2lGZgLgk=NjktEi+Gtw7c%ShI-Vm?HMpK-JD4-T2+Bpfb-769Wv}=Bi+Ar|zIp9UJ%5`gS)P>FLm1sG9)5V}_r87c^f?rOuQxpp{`pT~VYj6}@q7f|(W>062hny53m;);M<4OcS0C$dH|s|~PsgL*4elS^r(X~Kf!W~jRg_-r0CTYAUsl6suLr-ZxJsPbHNM+k<bXoMmuUg3p2tH$BdmK2rEYt@9?a28<MJxYFT*&`vpl>^(nT=OqpLN-ApSV-PpG{`Tt>+nW>}PYfUigE>?+QKK7CD=UQ#4!QAX)J#?J$f+Zg5C?Fin+gulh;{rguhus3cS|K%|tZR+_&u~sATKc(SkS(?Ino&t7cyVBA8TonjQRpOI6c?a0m-UH%dqxmSk4rnhk7?)Y@(R1LZ(I$r7nyq)(x|;gH^Dm&_Vv#55GV_*DC0^iL^~)<WkE(o#{k~%ei?pc05Jt7hV6P_y0|1&ch6AJV=j*J1rujPz#k)$%6-)xV+QPhJqed%Wp<bw3s`@sXgr?&1&~{YePg$Y%)v*@F(I<mV?Tq;f2zH!a#354afJySen`Mh@Zd=GS?EFjF5fkqNE%4<4e!x_@%qS`2O%e2OsUnqyYS`A<<am7YwRhjcKj+rQnrGQ%0{sKA+Lly2TL*bOD&i=guYziRGyic=CgnQ*m(_9q$JyCGKRLhu<j3N^YM}5oyQ1`Fu@|S&3{VKZJU;sI82&r~yrLfaR*!kS&Y}hM_=jj+#IgthfFq#kdyl<`$G<xP%wBjif05018$5edZCYV!R1XmUVcB5juQNsOMyijhFyf*0dApA0Mq?(f6Zu{P#T2@Y6jgsM^XrM#b(zO8Oa=>$uqse7I55;ytb!i^&j3J3=skaghem*PM<4M73;d%ata5aOWIgDgrCAvuN)~XcPA0`XPqt;B=2`nvUa#Z+IqcRE_Ph=3PU|p7DDyxN(8niZU0h+g7knSBcQFysftpoJ!+eXUUS21zu0BrlU2N_GLGKYzp1_|khnu8KE}~Kq9$(XYZwjE%JHy8{vyjy4los?c;pLN`5CbD`lOdwCU<j4p#yM=C4XlqxM;Dy6-iJ6(mV8cj)#bjTvV=|)#rb)c<Z%tlEMW9`yXU|d#KkHsfaQRy!7?uAD?9~v0MZaBKq?dkxUT0wN~0V|@hWEeY?96`Y*X)Q1+Tp=lCk^)&gR?G)3+YC4(BMqmiRYp5lVeXvK<l<?sg=@Ft;TTpNis09o}^KcY7W5fpY<tCD32Ff@%gRY6(M81Z(2qDO19ME`<l)Wna-QbvO~&uMq*}abDuzXjiWAZ<)h#hH?7AXMzT+|B(g=j90Y5fx@V;AA2-htT9Set2uaVq;G*}MCS{GWf8A6z>sjP1#CQ91?x7T@{@pQwA(IVQGy!8oMo8h8+iBc-}v9b3J;$mf&g<>Vau)Ef-Doaz>jtr29J~xXxU{5RJw=)oIO^Anx0JsLR#7Q8qOu89=ODU4{|tFG^1eD;5G+FMcFV{QBsD`ChQ|1xE0r^H%|Nkpn#lov{)x;3~~qTOP~?lO#z+4+Q>-`$QUlyi-2mK`M|3C=L6bYVF5cSE!Wpmq*{I2!CYyT6{S@}%{0xfpk+MmN1F_2ah4`?UUVEaeA`4HgA-Ui8I$)_#ZmwP&o-l7njjqt#shEkO<zxIlmhJ!qml?EPGR7U``!?{)XXsT@s&3^_MS}j^i`P(!aQmMUI_36-UH~=IUYUFfPO5ptMq6V%`da%lE{9%D1b(7q9jFP!hDZ6+bu4HO)|)Nb!=PR(2@aF&C*3dUF9I|ECPHz@E)k~6V!@z9B=7;-#jJg)DUt>1AxV!XZgah`bC5iCP=Nj(xjmSkD0Imj~({Vz@p@OCdC$cbr7?%vY6702A=9JTqOC_SHBCtBFpbr5dhI6>6!;ZSP$zNA=jBi7b##&GB4<qM(PbsY=Q6slElDWU22j%i!YM24T<!cPPyDyJQ)TSj!S5tzyT<J4#z2=&QvgMSI&`O3lBRzJI4o@8_b)Y9B_eS7dpntvplEkhnsAH9|&GRn_=Q}I{B70Sb8yl666HxCsY~<;&DTgX?<6>yb3oyc7ihGHcnCKZf_nS7Q(WFG?azNNQ6q=!F@1sWTgcPmINl=#h*jI??9R@J`wRGM4|}$AK3keH3o|22K#g!UBif8gREwKZa`tu0AhnHbu=vqe#Xm4n^dN+%55`PE+DrGa{We&%pwYoJ;T4m5w>*R>`&MN>tuyPU-b{i@FE7*77*{|w(o8;N(vi12wb7G8pOvHQRQz1Y^39UbD#j-`aU#C>!f`Y4mbq9h_s<JfGo{Bi=qM;R}t_yKuu;bj6U{=g8B}4e^3Z~L#R6ultyMrO$y%eFM5YX%>94cYh4oA;^!mK6TyB{CtY(0Oj_Egm9-FXo`O2ag~FXUiu~%X;y;xXY~m)<pbRB+4T9q_i?#g6N$BtLk2cBF+%TWpBA-T6eJTk~Oyx)G77H3m2*#loqKg=KZ{90u40Nto%?;dYQJ{#k1TDN!IMPx~MH6>6`DQ#mA9z{;0ved8D5j{aRjRr$L@iSo9ssLK3pm}U7<AxP0?O%i3j#480&BL#L!rrpG&cB9@LIDbHIg7;OX&C}gl?m8fY*_d>#?MvOyV{UR9JS+On0K@Bu~yNbXv%mkP{Kw__#?C@$=`h+HRoimv>t$^PIV^($!T1BnW5?gX<q!7Lv53fTDjEwkwnr9MW%mVwf0^Q~j&o26VNJNX1d(yE_3T<ha49xA5?HJdUiL<_}26&UneYyRQ^p{Ibz}r4(pjF|@UtlR;?oEGl9&2c#giBp>4E$edcTZY#lAmE|^6C%{w=0)2*Z6H`<ubZS1ksSm7<R-r2G16iNaa)^Ws4`|U0;&0Tc#H1-F*bk#+sUGW^D|pT$IJO;}3V<p^wH|=&Eot}Q#gRU9``(eaymupPX#&f78{P<f7s&;bQVPV|(Zw(K8m&vosLVFNUeB(}xCp2pR82$U(*1xeX(Keq0Za>ObKq@|98SmmJu4%POy$Q2Y<t_RNIsHa0WXVHbn?w_r>=pwv|T4<YQ<?yz8*gucwe9Vj%08^Oz*M&{Mh^Lc-(8-b9S5E6mQbDd;b<x@Bl*eSv!9Q2BFJdJ9TqyEk^th5r1mwWuNhuIOQ$jqh10}-4J-ssjyug8|}03rL*BWInV&I-;0LESXKkS9*jXIcrZS}|2@F}{UiSGq0*^aka#7;z_GTZp#gxvGO^{i;z+rXv#Npfeg^~2vUtUT-}p$0EjGW*;8FU7XJbj>gZkkhKb7b!b$bUPr&hirx$iPj)!~es&q;01K7eObr%5|aVE)GYBdCH%RYD4EI;0f$kBgE3hayH3QobvFi&C^C#pn)FFaw=hp*<!z2;j-1dC8Pf%9*<a`2Z1!6@I8zXp6M5@0v?@k7o-^bg!E=slX5ROfsuZ^(&JxX(55QAXWL^!JPS0q1X+hR%n=G62ok%^f0x;zUv%pV@#-~a3xEh)?YUSqI%#zs%$h>zle*jp}<<$y@TEGlb5-Jrb4r*{gHNm-bHH|AinMo81<(<_lB%r&vkAu*U?2N1Rziy*d=Y!jx(%jj5_oWtb1E0z^w29TJ<&!(b1kQPY6nk<N&D)h{t2(A$#VbtV@$Yuc;83^bX?4MMn`Ttx2f0CccUPq_*$5w%%cL^LIcZx3s}DI;uG?{hOwodpq%N;Ki+P&D?GoPR*oPIn}W>5p~)so|5Vpg;UmLax+)39pVsOqHKOoYpxxf<p_)46Rfv%?&=k5-HR9pwj)jIa<>kJU9FfQ(qX8!0TYUB2TEX6vG3EIqo@t`S^NT=^|hDplnV&kUO~Hb^6?{b2#>VEa~01ow{W_`5daHy?XgN&tI*X@XavGzBP>Sjyn?W%m#T>IUhF7RcZyBnIfWFHivtk-O$0U?m_9)dY}82+E@5b<9ZPM(%3L0Ro{wk9?TBkclmaFu1=_>l{8|D>ho`d!gVooF_v|LhFUgMs$9LrU0=>{4nVQc6ZPMj6sR~u!A+$~`LM|{s(MGKuygxfD6nX#@SuuWI`3`Y8hmpXqmKs{A9&QD$YS<&1u5I;;K9CwwP#Nvx3X@e8i-wR_6IP%T7c9sBFs%uIGG&nf`z`|6wKi~o!0EAgggOa8&NRNF1yR&9%dbZU@#gJ9Ba(@(VS+)R7N>#(un6mH7WnrWZw^Bjb|omf#*xQp@qp#JOn_ZPNTNyS>|C2))uI{8S9Q&zXocOfNn6Q9ug2UFKi^zbT39t8;bP6k>b@v7mQx$EnmB>|Ss1~`q~gG!?Y>YQ^#(DxPiSHdU|`#R?u;ZapC5hq?v=Mm3UEE0uS}e%kBPZ*8iT<BAZam+HTgOR+r;FEIclpLQvx7SdE8gL_Lo}~l+=W>RuoYSE-c+aJ2k3mpi9*T5>9Z2iBK&B<~kup2QLJ)q|g^Mv9Q?+*!QwHuuaSbAyP{Ejtr(8R6{^b&det@IgrCR+Sr}5wLyIvp_64$w=IzA$ed&4toEc~h%kK!Oc56m!vvy|hW+z9fnpUEtD_?MC2qsXn%HKdPD5TME1TtX`1#oL8}F@hO9EaU?4zQJ5z8Jpm}ZZ>3`D;C3MZ<0f^`hUweS`|l-J~h!0T>^dE__=j!4=}7$~2SNnIK>bnsMSK*ZW1%`ZtlP1XyfnpD&+ui3p)4I2#c+Sx4{>D?wrD!|Jk8>&T>1TNthsE4t3Bh}xx87<j3%F*-)ZKxyb&JFO+meI8AYav~#yLutpbJi^^I#FMvPn{IhXn^^zMh|4*d#Hh}vgBUxLLF)_LrNj-v{8$oh7C*^lxnI*N~*SyC8`!tsL5x-x0}A~RYI#}udAJ<^)A(Ob1RPM{|$D-z0axYzzcB4@pkXj%y#T&`(s@?cSvG&27NDcWy@lQ`<yaPh~UKysr?CP%K<#~*Weo*gW+X`{u>T5!Y9Bs+Xm!kho;Pa?{90U_!Q-fc~WV;vtn^<9<Jw@nXkOCLJG~v!zG-7=-=iyrhB+D=l*^1u2P(Lh|1ZiDNps<)YTkZ<aWsF(vtVut%L?<srt)%b%Z-|;aU@Otqsfa%N^}St()M^iik?GY3DUWawL~Gk^A(rB&bGV*+`j8@C`FCXr{4OpjDruq|kxt&C-%6u}>*U00r4=0geK!279v7bUY>H$q^$w>Y&)H3X&Z-6D&vtFfWzGHO<oDGAiL<XT<Z+4H?8gR1X$f(;=!faAml%B#lY|T^Tya0|9ABC4i6}GX;LnwnU2tNE$T^-)hgP1~GmIP68mb7b112=#)wfF^6VJCo(p(EF9gqBWn_lS%QR_#yCaQY?Qq99gT~#25W_7=;pX&SbESdk&Gs;U;_V{T5y1_uE`4*(u%1{XXkeSvM(BLqlccsY*jsBR0!Acb)-i#2nl)Z>yQeY7)<_Hy&gI5c%=P(n`ale#Wm`_g{YNM)Ndg3V$|XWbjd&`*Vmwm-)&%nAYZWMn#7z4<`Mu9HZx=y;tG3l&x9c;M9VM)hwt@rz+R@X(V*aBx5iDn&Mx4@1yVM(1zGgXLlvz4-j_AT!2&S!2aILeCrDL=wO-Cc%8T47HW*(3MKHJ>9PUVu79Wfu({t70v~yZ^lR7>Y`AU4jQ0FIDBsj@&Iq2QkuB?za#$Ctpq(N=Of|Q4sRIAro44fk_(f?0^VMk!!yo5s40xRserUf>Zk0ue2QggA}Yzwe(Dk!01(gd!rK_XxDi!R%huVF3K4B&G_qkK_M-K{aepo;RYHsRfAq4w)Ax0<MbJgg~2Q(W!Pm;Np8qyo;CFe>IrqMX>#X20GQE5j^OEu)QBHkhl$WT$(zSF6+Qz@tbJ9XuJ8g}ixK+WH!eU^V@Ny-&WHddJQ00?#w`<GiI<9sFkv@@=Zqz#eo7u@qpt-_x8rrNx*8&FwiA`c0p1gh^9KF-byW2zT37WE@71dc6|<Ax^E#O{0Qe=EQH%h>$GI52T7cq)p|)tm_eTKafqQof!^l3Qu#hEsN_IPz+&Sm<8|!q$L3AcHm%Zm|-+k$C*1dCwsaDRZyLOSc3*5F==Nn6owt%V52RwnY+*%VM^^?{GS?!CY)}>Yt~N1X^oh->=AwJ7)Oz^z19z+E>hDR=aVMAJYWPWt{Zu)y_Qq3n4(iAx-xw;V88R@k#;F7rVp&Bre23@d2lj@ieG;->PyVi^!OpZf*`aIY)MmO{RZxUr(RpUND-=ZAcd~oH1EQkQWiY96oMA5j)V(to(_dX!&+<RFd#nfL!H0-fP5WkN66?GFxV<kbsN}zP{mEBz$#w7&iWOjN@PK;9fOz@Z6`Pw9Hyz`9Z<}?R(WD_5>R*|29`7xHGahLY6ACb5o+Dz=WGLN0vtRd!&m*#7nzAMfuQD#ZU)JL^q>b_y?2I>^Q`A99DG%lwRK9={E?hve?((v=VYpQDTDt89bd8_QJ6#*_MtN-(u6GS7n*>In0oAuySjZ&g&a`BPSf}zszHu7jwMV|dI7>v{#wUVsr1+iF=`G*%C!q1*c$dJBa%?VN}cux^(Nqm5+oiR-=*n?{6H;MU~g$`AFMw8rr!YURdl$ALx5A#DBU9D9T4Sm6<vg`5FzS{gOJ=J!q9PU`CPiIy1c2H0Zu{VnH5;)=s*lv6HiCx`DVl-vOa%{m_^@)DRg7S(P0XtbeUYVb*1QKn=6H*JIO6FQC<f^nNTOJVss}u;8h~SiYhb=(7-1QL21Gr*yL}oAtMVpj9eEZ#ssJaX);QTROS!zsJ%Tz_br)=6vTM?;yNR41C}|XGW8$O1d?FE)voRJmfP^C4UbphP*0C`)qcm=26|weOF%g)k$c7+Tr&<<pPIUn4yTNRT`O!?2JxDZC#7+}qg#c0ZJueDt}d^NrmLfPep%|_E^2MSR<8yBI-79$b4e_rR9RD7DHfO!(8vmK^}fN?s1aO%1K*ZXQB*0t0<u0i)}C82<Y{-70t>dSX$qtiOJ-GgfEv>oHh%7c8sv@HTI!q^C?)FX^Cd^Lhd+TI6SZY5?{B;}cpT2@#NwHOnIG`w0w#?*X0AOU?`>vTcyWLb!r}v9cNF=$O4!F6J-jJXgkgfKG-PATNLE3UhSp@|VRC47MuFUcmX44KO6Tz9G7jt)wrae%{b(`UhFe*Eo|^e?&QX7|)h#B%kJ<{fs9ac$TxpB6P*ugL?$Qt?0OgX3EHsM)>c}%v{(%i<;7A=vtd)pG{FkisnDNF4rRNlcKwt602y*#>pL0;*>_B0LN;X43HMSJmq<r3<&k3ukPOzSt&M@G<U5+I51wAN$2pBvx`y~G!!sA*(0)FEO04PZzrskkvq=E**aCoFPEqKkq$v_*ZkkD}~i<`LC=qaY3;3yEu=dl4kSjorgB+fPfIog4hyDhCMx>xGmYqw1Mi#AsW&=^jwV49;yvaU5@2lhMTeKq>Bly6z-&e}MO#-VE4M;l(Rjid3PX7o9H&Z+>gpn^3zMt*Wb9^WdjW`|CI6R5HLYzgqa!|916GdWa{udJ8v6wd(e@W22AW~ii6U`U1G0qUV0YX=9`+uiI>Nau-tT3=(zK#bsF?*wi65!(&VN=r;LYdbgjgc`g+F35{YV#L~izrtFJegpd^G1|;5WOGmyDxs!6O6Rj=l&&|UG+B)<vJbwShisbv&>>hIm$*`B_q4c6--fBh#+tQA8gwoLhylEu0~Od&;T?%E_DQzaSKJ)sJ}JR0t6VA7i>?KfT((1nLkEfjGyw6f=#UM}5e;NG)|xY{I@SrnJEt@R;KUP^+O2zBll^r<jXlytvh=!;>>oOiY|`!5`S&HKd{H;>FDtKfP$gO8CsQdhIV>*Io%P9)u1SKL*!`uxeY>M<%}h_2VF_8fdINSsDKo-&-#OKuqpBVhHSQc{Jz3O2xuEWlN~zj(^2_k&cIe*5hTiRXpTg~+Xw^GrYijY1bH%@k2O(s@R4Hz#Ri#G7P=dn~K6U1txnmIFAqFa5_PeBya>rg81e^)CBm7b}swD)q1bhtGBL}d%K;}6R<)AyR!x=I{dh{eEWjMJ#q82nQq9tH6q-q9q_WS)k1X_*~5&11rp<PFjgPZzyAPXck*RW=4PNIoZn6<EPd(hR7Ip&e2uJ~(eKLNES6=zC}r!pF{Wn6?BvSXyfh_aZ+$h~UcJ77$$Z1%PbL(59q4t4HzHT>faS-HmfuTS%2xi>bb=}cx+xQjm-57>@Qx>M_Sa|vvVS^7Ko3RG@)#^68E^G-bDY)I6-2c%aijjJQ5+*}-8odV>gj_I`0ccE3yAEmunRlwmmsFL|Zme7!Qti40I@SfdlE`$y_M!F!4hRn_eS}0~26@+0p?71`Kui&Y2b%i9RRRViR61S$T!b-%6qxmXi-4JkKp}6BxzTx%S?>1N}pnAW5_vVN9FW!ZxFW#Sqe|q!$g(-mIP*8TmQ8*ZVmG2XMl2NM46*~&T5Ha#O<ttQkyk$tD0_kAmf#wUT2bDI~8LA~=c@C9f;~=SF&;DcXvyQbrgHe*3r10eS={Go)F5-_qNzP+)@Onvbfr+HJFqh(PbkRN7Lfku*X^1blUM?-Wrl(kd(VZHWg!W7Wf(?KHbjgL0Z3lklm~9|2j=<-c%IN_z?2c|9fjUPCmD34w3Z;`pZH&rE%U$^e>vGhYCTC$)&B8YfsK%v}flb<VrCbE+lQ$a5Zbg9g&wTy_NND`-%r*bV_?mxV3L<x+`G!!68LBbmvg$$Mhk9GfA&Q7$7TUyM@LX`X?rW7LZ6pu@O;`+S9u>XINwL*ofWa!N&T|3oWM7c8Y7?zcMoaqYi^ed`QnbD=7YfS-DF_?&bu)pqR7t?JSNmt<bF5Vr)cA#;8hETwK~Py<SjdcShq~n1@c8^p_WF5WL@R|*+RR;J^ni~8wR3DAABXb{juTyy-8ixjH4nO4(wX1$jYxL#8P^e71tc}-AhK-`&Jho(-3b-|u_u~1?@tjp)YJ0080vbOq1GWO<zX}f;7(@iD82NBk*YD5+6QvdM(oq*1Ln8Gyo*;+j@iJQ;bgU$R5ip)p%Psk&7HTxL~&4>X2HZy^S}Oky7Csg>tFx!UrX<D_1FIe+R62=|M73v-v9Y;|M!3X`j>y5d+D!#`R@sofiJVa{*MGQM*aF9|GOO7_ps;)7zAexC&|+Ttx*v9mBX$Vs-rU5gLq6YL4Sp2nmAN0F_h#Ho4%oCnrK=(qGwQ;11D&_r6BWvvY$r@-0F~@LP2i=Bwq3-84Y+k#FH~uNfyI_evb@i?P3=riI=*##YR{AwhZ<dsa<K|P*)J2dC&zKG;%Dh?s)L7vmJ#$t#**QM3=BA3;EmOOVo`We=~08u&ON@t6VB;idv<bq4qu+?5j(Y2A-LuzMjI5g0e(1Mt4xi$YW$`)t8>BwzN6VQTHmIG13&fEmj)s5~=<}JbO;xet@C$4#=?Ft^1Qc{jMbNJWp{8b0jFQ>9<D2vXSz}tz44n#iiOlCX@s7p;3j%j7h6pC7`{FK#&cMg+{3c6Jj*nGFs*~>MQ5eVRex%<Bl?wJJmIgI-R%zJ{6p*(Qq2{yXIkGMj-fXVr(BSj(}_If+pqKJ6HE46<JhQH-OnMkqnepe%(fLD^3Z?^JOcu;#;k=OG?m#5$h~$OoOz}7u#^YS%4oGvuS_8qNfnvthl5z#ShVobAG@d$-+8T)Z9!4;+$^fa4_GUC|8|h)meLP@aDZ*1`%$IR)v$FfHo3D4B*Pj<?x}uC+u49<xdT}{(=#{_M(5GRInJMu4k-;ejUEv5^o;eFx|QxZPSZtwmdVe*~gn}@L1hkf=wz=4+R_2$?z0rm$euL;CzINL42<<i~#qz(RSaxcj{sHL>=JC0L?-1@YOwzk30%o0T+ISE>8UWgjn`w+;(>YLOny?Jgl!rHG`Xtx3d2=uY-{G6thbiZt+(~)|#7JXI7fM88X#^+e$XHkNu>Ssa<+*GY959rB!SdM~t+X_Zs#kXVfq&s{|6Z!tH$0nkBYYP^2?gAD@*3QS;z`KN%2>1pVzf3su0sQz`i=y7lU6Be}Y+U9Z(C0JREwZPHF7>|$@yFDqwuz4kW;XAXnR+7i6PhEscU>oc7UynQ2H91~N64+Y&bAa@IMQV$P(CqTwfI@u#|)XU*nu8B2lG*+Cr>L1hnk`76%*)C5PD$raN9r7&VLsjgcylYskYyTSEI)XfdzPm*wiI2#4AjtiH4*qG<a(CvxkRu*SS0FRx0#&$l^N_GjV{l4Zq(N0LZM{O>qJf9a=%srCl83Hpt33-{&-5k8FmBvfRzWE%KLtZgpDmYi8L1!@MQ>G&@aq9($`P?14UWAkNtjA416R`K3YQ!hbYmD=ya(j1gxNRX;e;Oe%$Rt`w>0PG(pM3uV}}i!reLm#%Z}uVngJ#vFO?<fV7zNKGenVkUp0nVvNWNu?owrMJ_~cmdd%vknhCiLJ+&fABSe<h)<>`$;x6qEOwJu@wC|t337>!W=9#*#K}*2gQbqoy)9x0m>(1zKX*YxL<lXKymRgMm<JO-2@!L1A|8sc2G|WoRzWwfL`0U%K|D}D<g4{+~+xF9Ht0DhGxpz&ONo#}bJA1XAU6m>nJ%{itw0vEwqvIt8CBl96Y=-roQ6*rW>4H7DVLgXTBS0LHdx4wDXA#~W{R))oAN5sw9(uPGW;Wo4-zAI)^)2Mf=#FBdNO7RI#HZA+&cZ-<HO?3lF3m5|6oQF!&u>jrvDuzw9b4mgZ59K^Q*qUJ8Dpw4<LU7okY!BoMFnCBLQ<{wo929ct*(XEaFx5{iQ2&HsJt8=kH(`S8;xOx0l$M+!*TiBujzi2>WvxlF12TQ0<Pl2lV&6Q>7>z?`JvuqS`-hUA!!C5fx-{?+cy8&`UubX)oS5lw!x7cODT9h)=hl8oUwM#9ISpjZnXAi6qnKUH?|u6s9#rY|9KIiWzH_A-W@mFT_zvF<N{?M=(6;xn(m+R#?Ty#oaiF->ZH*c*xG@Wwu-7(CkEivi#ovP7#m?$#1}?2daPI4f*UQkUnP0gz#jY4u!87qbx4a7hQ>?Nt3pVFHcX0fQkAAedAef#K5mCw-5li@xTfg&t5*RxQA^;d<R-pSt=Rw%NCdD(8@%7Rn3`oZ3CCP}CQXp<DyBoJtGK1cloAa(dz2C00XPQ>Hl?eCP?BUlf_tKY<@Aw?Vw5OeQPB86Rea}-9`IQXa4Oa2Hv3M|B6gejlTFt3oF92Sjw9Wzw*%=3WCUb>!IcxdS`gvX-RL~Yem#OIh8<$#g@i<r%{1O6i-0V&V@J}4-6Cs12VeTmOW=sW!`H4mAnm#H(dL?dmE=3mD32=L0s_L}VvQk)*pDmKW;J0f0nizywX1;Orfk?~=|)}nLRJC*1-~75&tLrh<<r;UAKtxrefr|{b9C)ZF?=CcQG5XKSI@$yuU<`kIO2P@MjIt;a02FoFo5KqB&`-Y?9Js#EiNn&nZ=0^F*q#XiG1Z5trQUN1W7E6l5bzWeL?R*3NzooKYjk@yVJvOWJLB^^na*>g%z3lqE;gbj}$|2^3G%b@MZz3Qv=?E`W27O^;1!De0Vc^w1|<2z_{=}ZqPe}JU}UG{m5?)mtx?ckQ)i%gJKc_Up7t?(0$xkke~s*#@J&;L7t@%{_NZXTwc*PRAwK(8LS<EB%81((s>afGQw*e6Sx<E*f}|`p`?k9XGHRRrE@~UG5q;w)K)?Qlp4`s_N_@aYDbEbuu3^@#X5S<xbnFhDw?=8x5F6`Kn$@bFK&Mz?Vq=9i@RP&+PaH6cgJ0OX>6wUP4Zyel1dmD8ZtMM4+vNlqntD)&*c5&$KrnQ<NL4r(7J}HjW2CEf~(>c3ai9J6p^8Oww^{5E*cywPanvo0-7(sJ)N-Nmv9IC+<q%zGYr`sOVYKwUT1VZ3szRBhPaB`%E{*#TcOBMze=gg<ZbP+b!;|%q}@h-mcwD14f*X5WkSdrnNohB`_k(9pwbN0r6GrmbxxjA>Y*V$38$nkWjycTtu*~|R&?cx?4VwFenak75L!DpmX@w8t6EbAB>Sy@sPbNU6WGdoPIP#f*1rD*WlSBh`&mudqN?Q4!Y~1?*tcV5@0$_$9NEez>kJHKka0>9QdNEkQ%sT7o`mWU{YBH!Lwq{4zelGph-!O;cOxY8Sn9r4k7{J>X)>)^94Rz>qrNa|bGZ>X(S~x!2e1<{9S$;+aKG}E-(3_^5hDr@7x9N-vDyrG<e=GICyzIvet{~p-354`Tnx9#c3&Ok{V>Fg^3&s!hvUKT#^cJLty;zk$J<)MvO~4xxb5mU38QzQp^G8H4V^zY@%J|k6s!Q!Kg&K=L$eU7*@Su@%Du7d8Ddqk0aLljKtaIm3-lQ9V|bhURzdWsnMd1bmXHD0zsP|112zvKjz$-9Y_%K}fDQrwE}?D}?|<gMe-S==`s~{m;oGOD-)jFAM0+-v;}#uhEhdYC)kP4_7EfRP=>>2Rih2Q)=5r*gLhGquG$rkYp0>!zXp1_tz>M6)`?hf$TjYtu%PWZWBXWQiHc6mp@HEffy?A>1B7FTO{Nd&6=Wl+vZJ@$bqk>XwiodNIgk!e_Jeeu~9mwKL`LDx+>eD@<8s0DBjjW79H3#ASq}E`)gxua(Al4<Tj9)^c1n*EVF;)NHbIICb*{Qm^9VOXLnt^^5Dp4M|Mso@|y2isL_~shl1T1F9G9C?>)gc6nv|~SD$x*dSL1@1er9;zOgePH;`SzPWdGH|Zn`tuCX<2hUh5fuvE>>mzgu5?@rj_R3D_=^{m6N7fa)1<)=m@8dYIBSL(AHK|GgNj~!Fe5n=D=d|RdFA{`q(rF>Gbg<Cur^4GI&g8$0yX#P|y4Ek$U|Yy|bS1<cK4Bd_=E&m&1;_3>GPTtvUZ-woX@}8%05DqRh02K%;IR$-H;bZ!DzVaaCel=J3eTwr*v(J<x08)5p<sq`1aGX{J%9e%<TsfgX=tv$)GZ8(XkrsE#^ceB1}2ih|iA)MQ`?x#wbTl^|I=<N&1<TL?qyIn&!{<aYYDGAS{0%(_`)Go8#Q@yNEr>GT-iI1<#JxYEBi^S$Dg7_hH^Pgz6IMz*G)YT;5fEknb8mOk<nv!JfZD3BjVxtzeJX&DF0bsGL8@r+IpTTB0<6Tw<n1~kL}Bs!##IJW)P03-Qd(a0;Lqf6_ek$qiIYg(c1jCF?nxHDs&+~-)&MW+t9FLFlIa->c7q%yWuK^Bf$Pc>w1@>F}NsJJoS{C^=XcVO8Z)Q}$fccTxj3F2cFS9fs!ZUY&trQeg!@PKFr`L_bB&gk9@helQ|4MdMg%+(VdSEPq7TWfQeG+a~<3lM3=s5_`ZErRI`FtGzK9AFT|;mzfwcFisOYVakuhz6Hc<TLpQq3?*g@t0yk(LJ@*nB{brG-x7<N&;xNCep3Bq@^y_alFNxTLC99IRO??$JlgTs$N^OV#;9A090{nQzUHssk6PmiTmjt)Mj_T|Lsm$9UXNWi5(qf+>2xeR(GcIXs|5qY<x|s^#1_$RE`J'}

EMBEDDED_SHA256 = {'audit_snapshot.py': '75a6118be7c78e17a4fbc1a2c2fd50ddbf4616bdfc5750498a7afda7c808b558',
 'corpus_runner_snapshot.py': '9910961f79e64f8ddce50de94df986f797b929cb96e8738b22e2cb25e586a05e',
 'crawler_snapshot.py': '8353e2733456421b2324172bbc84e5c6da8bf58717119cb3c555421421994f7d',
 'diagnostics_snapshot.py': 'b544e20569f23328c4df6e6dbc07cacda9f9fb6921bfec77a3fdaec307a2bcaa',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': '1021c7ffbd10629fb3137d0102c8bb43106928d785bca3293125f9a199b55604'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py'}
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if sha256_file(supplied) != SOURCE_SHA256:
            raise ValueError(f'Parquet Input không khớp {SOURCE_FILENAME} của revision đã chốt; cache hiện tại được giữ nguyên.')
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Chế độ robots theo `RESPECT_ROBOTS`,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
`MAX_NEW` giới hạn URL mới, gồm PubMed chạy Selenium trực tiếp; Selenium tiếp tục các URL lỗi, text rỗng hoặc
QC lỗi đã có checkpoint nhưng chưa thử browser. Chrome được kiểm tra trước lượt Requests;
nếu không khởi động được, ô crawl dừng và ghi lỗi riêng, giữ checkpoint hiện có.
Kết quả cuối ưu tiên engine có QC tốt hơn, giữ Requests khi hai kết quả tương đương.
Parser 120ask giữ câu hỏi/câu trả lời; cnkang giữ thân bài, bỏ menu và bài liên quan.
`PUBMED_SELENIUM_FIRST = True`: PubMed chạy Selenium trước lượt Requests của domain khác;
không gửi Requests tới trang bài viết. Đặt False để dùng Requests trước; `BROWSER_TEST_MODE = off` cũng tắt Selenium.
PubMed xác minh PMID theo URL và lấy tiêu đề/abstract tiếng Anh từ DOM; bỏ menu, figures và references.
Trang yêu cầu bật cookie được coi là challenge dù HTTP 2xx. Khi cùng mức QC, ưu tiên nội dung đủ thay vì text quá ngắn.
`records/` là kết quả cuối dùng để audit/export; `requests_records/` và
`browser_records/` giữ bằng chứng của từng engine. Chỉ nội dung qua QC mới được xuất corpus.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File có một số mẫu record/raw để debug, không nén lại toàn bộ raw.

Mặc định `BROWSER_TEST_MODE = 'failed_only'`: sau khi Requests hết lượt retry,
mọi lỗi HTTP/mạng/timeout/challenge, lỗi trích xuất, text rỗng và QC `review`/`rejected`
đều tự được đưa sang Selenium, kể cả PDF cần kiểm tra lại. Log `selenium_fallback_scheduled`
ghi số URL được chuyển và lý do. `records/` giữ `selenium_fallback_reason` cùng lỗi Requests gốc;
nội dung Selenium có QC tốt hơn được chọn và xuất vào corpus nếu đạt `accepted`.
Chrome/driver không khởi động được sẽ dừng sớm và lưu lỗi riêng. Selenium vẫn có thể thất bại.
`off` là lựa chọn chủ động tắt fallback; robots bị chặn khi `RESPECT_ROBOTS = True`
và URL không hợp lệ/địa chỉ không được phép vẫn không được chuyển sang browser.
Sau cập nhật snapshot, dùng `RUN_TAG` mới; có thể đặt `RECOVERY_RUN_PATH` để phục hồi run cũ.

Tiến độ và cảnh báo QC (review, domain_parser_fallback, …) được tổng hợp mỗi LOG_EVERY URL.
Lỗi HTTP/mạng, trang challenge và nội dung rỗng vẫn in ngay kèm URL và nguyên nhân.


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'pubmed_selenium_first': PUBMED_SELENIUM_FIRST,
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: nguồn mới mặc định `ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Chỉ khi bật HF_UPLOAD_ARCHIVE = True
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload corpus, diagnostics và manifest; không upload ZIP raw/checkpoint (`HF_UPLOAD_ARCHIVE = False`).
Mục 9 vẫn tạo ZIP đầy đủ trong Kaggle Output để tải về/resume; chỉ bật upload archive khi chủ động cần.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = False  # ZIP raw/checkpoint chỉ giữ trong Kaggle Output.
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
